# ☠️ Blended Backdoor Attack — Full Sweep & Defenses

**Attack:** Blended Injection (Chen et al., 2017) | **Model:** ResNet-18 (CIFAR-stem) | **Dataset:** CIFAR-10
**Locked Constants:** `SEED=2027` | `TARGET_CLASS=0` | `ALPHA_TRAIN=0.1`

This notebook runs the full experimental pipeline for the Blended Backdoor attack. It evaluates Attack Success Rate (ASR) alongside detection methods (Activation Clustering and STRIP) across main (1%/5%/10%) and sub-1% low-count poison sweeps.

### 📋 Notebook Structure & Cell Index

**Section A: Setup (Cells 1–7)**
* **Cells 1–5:** Establishes global constants, mounts Google Drive, clones the repository, and imports core model/metric utilities.
* **Cell 6:** Generates or loads the shared visual blend pattern (Seed=777) used consistently across all tasks.
* **Cell 7:** Efficiently loads clean CIFAR-10 data directly from a Drive cache to avoid network overhead.

**Section B: Poison Cache (Cell 8)**
* **Cell 8:** Poisons the 10% union set and uses a prefix-slice method to lock in the 5% and 1% sets. This guarantees strict nesting (`poison_idx_pr01 ⊂ pr05 ⊂ pr10`) without redundant calculations.

**Section C: Per-Rate Training (Cells 9A, 9B, 9C)**
* **Cells 9A–9C:** Independently trains ResNet-18 models on the 1%, 5%, and 10% poison rates. Checkpoints are saved locally and to Drive, appending Clean Accuracy (CA) and ASR directly to `blended_results.csv`.

**Section D: Detection – Activation Caching & DR (Cells 10–11)**
* **Cell 10:** Extracts and caches penultimate-layer model activations on the poisoned training sets.
* **Cell 11:** Runs Activation Clustering (AC), comparing PCA-2D and ICA-10D dimensions to calculate silhouette score, Poison Detection Rate (PDR), and recall.

**Section E: Final Summary (Cell 12)**
* **Cell 12:** Compiles and prints the final CA/ASR results table and verifies that all canonical checkpoints and index files persisted to Drive.

**Section F & J: Test-Time Alpha Sweeps (Cells 13, 19)**
* **Cell 13:** Sweeps test-time trigger opacity (`ALPHA_TEST` = 0.1, 0.2, 0.3, 0.5) on the 1% model without retraining to analyze saturation limits.
* **Cell 19:** Re-evaluates the main 1/5/10% models using a realistic, matched `ALPHA_TEST=0.1`.

**Section G: Sub-1% Poison Sweep (Cells 14–15)**
* **Cells 14–15:** Builds caches and trains models for very low target counts (`n = 1, 2, 5, 10, 15, 25, 250`) to map the exact boundary where ASR plateaus.

**Section H: AC on Low-Count Models (Cells 16–17)**
* **Cells 16–17:** Extracts activations and runs AC severity checks specifically on the sub-1% models.

**Section I: STRIP Detection (Cells 18, 20)**
* **Cell 18:** Runs a batched STRIP (entropy-based) evaluation on the low-count (Sub-1%) arm to map True Positive and False Positive rates.
* **Cell 20:** Runs the same batched STRIP evaluation across the main 1%, 5%, and 10% sweep models.

**Section K: Grad-CAM Pre-Defense Verification - attention correlation(Cell 21)**
* **Cell 21:** Computes Grad-CAM attention correlation (Pearson r) between clean and triggered image pairs to measure model reliance on the trigger. This provides a crucial independent verification signal, especially for the 10% rate where STRIP evaluates to 0% TPR.

## Notebook Index — Blended Backdoor Attack (`blended_finalNB.ipynb`)

| Cell | Section | What it does | Status |
|---|---|---|---|
| 1 | Reproducibility + Global Constants | Sets `SEED=2027`, `TARGET_CLASS=0`, `ALPHA_TRAIN/TEST=0.1`, `EPOCHS=80`, poison rates | ✅ run every session |
| 2 | Mount Drive + Clone Repo | Mounts Drive, clones `adaptive-backdoor-defense`, sets `DRIVE_ROOT` | ✅ run every session |
| 3 | Core Imports | Pulls `get_resnet18`, `calculate_ca/asr`, `load_cifar10`, `CIFARPoisoned`, `add_blended_trigger_global` from `core/` | ✅ run every session |
| 4 | Load Shared Indices | Loads `defense_indices.npy` (2,500) and `asr_test_idx.npy` (1,000) | ✅ run every session |
| 5 | Transforms | CIFAR-10 train/test `torchvision` transforms | ✅ run every session |
| 6 | Shared Blend Pattern | Loads/generates `blend_pattern_seed777.npy` — same object used across all Blended notebooks | ✅ run every session |
| 7 | Load Clean Training Data + Test Sets | Loads CIFAR-10 from Drive cache → `data`, `labels`, `testset`, `testloader`, `testset_raw` | ✅ run every session |
| 8 | Build/Load Poison Cache | Builds nested 1%/5%/10% poison indices + poisoned train data (non-target-class pool) | ⬜ only if retraining |
| 9A/9B/9C | Train Blended 1% / 5% / 10% | Trains and saves `resnet18_blended_pr01/05/10.pth`, logs CA/ASR | ✅ done — do not retrain |
| 10 | Extract + Cache Activations | Penultimate-layer activations per checkpoint, feeds AC | ⬜ done, cached on Drive |
| 11 | DR Comparison (PCA-2D vs ICA-10D) | AC severity sweep, produces `blended_dr_comparison.csv` | ⬜ done, decision still open (see doc05) |
| 12 | Results Summary | Reads `blended_results.csv`, prints final CA/ASR table + checkpoint checklist | ⬜ optional recap |
| 13 | ALPHA_TEST Sweep at pr01 | Finds non-saturated α_test value (supplementary) | ⬜ done |
| 14 | Sub-1% Poison Cache (low-rate arm) | Prefix-sliced n=1..250 poison counts from canonical pr01 | ⬜ optional — **use latest occurrence**, appears twice |
| 15 | Train + Evaluate Sub-1% Counts | Trains/loads low-count checkpoints | ⬜ optional — **use latest occurrence**, appears 3×, not re-confirmed this pass |
| 16 | Activation Extraction (low-count) | Mirrors Cell 10 for the low-rate arm | ⬜ optional |
| 17 | AC Severity vs. Poison Count | AC on low-count checkpoints (PCA-2D/ICA-10D) | ⬜ optional |
| 18 | STRIP vs. Poison Count (low-rate arm) | STRIP TPR/FPR across n=1..250 | ⬜ optional |
| 19 | Re-evaluate Main Sweep at ALPHA_TEST | Eval-only CA/ASR re-check at matched α_test=0.1 → `blended_results_alphatest_01.csv` (**primary results file**) | ✅ done |
| 20 | STRIP on Main Sweep | STRIP TPR/FPR on pr01/05/10 → confirms **pr10 TPR=0%** finding | ✅ done |
| 21 | **Grad-CAM Attention Correlation (NEW)** | Pre-defense clean-vs-triggered Grad-CAM on pr01/05/10, Pearson correlation of heatmaps — independent verification signal at pr10 where STRIP is blind | 🆕 new, pre-defense only; post-defense pending Stage E |

**Fresh-session quick start:** run Cells 1–7 only, then jump straight to Cell 21 — checkpoints are loaded from Drive, nothing needs retraining.

---
## Section A — Setup (run once per session)

In [1]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 1 — Reproducibility + Global Constants  [DO NOT MODIFY VALUES]
# ════════════════════════════════════════════════════════════════════════════
import torch, numpy as np, random, os
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import pandas as pd

# ── Locked project-wide constants ──────────────────────────────────────────
SEED         = 2027   # DO NOT CHANGE
TARGET_CLASS = 0      # Airplane — DO NOT CHANGE
ALPHA_TRAIN  = 0.1    # locked after pilot (achieved ASR 100% @ pr05, alpha_test=0.5)
ALPHA_TEST   = 0.1    # inference-time trigger alpha — DO NOT CHANGE
EPOCHS       = 80     # template standard — DO NOT CHANGE
POISON_RATES = [0.01, 0.05, 0.10]   # 1%, 5%, 10%

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Using device:', device)
print(f'SEED={SEED} | TARGET_CLASS={TARGET_CLASS} | ALPHA_TRAIN={ALPHA_TRAIN} | '
      f'ALPHA_TEST={ALPHA_TEST} | EPOCHS={EPOCHS}')
print('Poison rates:', POISON_RATES)


Using device: cuda
SEED=2027 | TARGET_CLASS=0 | ALPHA_TRAIN=0.1 | ALPHA_TEST=0.1 | EPOCHS=80
Poison rates: [0.01, 0.05, 0.1]


In [2]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 2 — Mount Drive + Clone Repo
# ════════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Aritraghoshdastidar/adaptive-backdoor-defense.git
%cd adaptive-backdoor-defense

DRIVE_ROOT = '/content/drive/MyDrive/ps-capstone'
print('Drive root:', DRIVE_ROOT)


Mounted at /content/drive
Cloning into 'adaptive-backdoor-defense'...
remote: Enumerating objects: 298, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 298 (delta 10), reused 2 (delta 2), pack-reused 279 (from 2)
Receiving objects: 100% (298/298), 164.83 MiB | 28.52 MiB/s, done.
Resolving deltas: 100% (98/98), done.
/content/adaptive-backdoor-defense
Drive root: /content/drive/MyDrive/ps-capstone


In [3]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 3 — Core Imports  [DO NOT MODIFY]
# ════════════════════════════════════════════════════════════════════════════
from core.models      import get_resnet18
from core.metrics     import calculate_ca, calculate_asr
from core.data_utils  import load_cifar10, CIFARPoisoned
from core.attacks     import poison_blended_global, add_blended_trigger_global


In [4]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 4 — Load Shared Indices  [DO NOT MODIFY]
# ════════════════════════════════════════════════════════════════════════════
defense_indices = np.load(f'{DRIVE_ROOT}/defense_indices.npy', allow_pickle=False)
asr_test_idx    = np.load(f'{DRIVE_ROOT}/asr_test_idx.npy',    allow_pickle=False)

assert len(defense_indices) == 2500, f'Expected 2500, got {len(defense_indices)}'
assert len(asr_test_idx)    == 1000, f'Expected 1000, got {len(asr_test_idx)}'
print('Defense set size :', len(defense_indices))
print('ASR test set size:', len(asr_test_idx))


Defense set size : 2500
ASR test set size: 1000


In [5]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 5 — Transforms  [DO NOT MODIFY]
# ════════════════════════════════════════════════════════════════════════════
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])


In [6]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 6 — Shared Blend Pattern
# Generated once with a separate RNG stream (seed=777, distinct from SEED=2027).
# If the file already exists on Drive (created by any teammate), just loads it —
# BadNets/LC notebooks and AC/STRIP detection notebooks all see the same pattern.
# ════════════════════════════════════════════════════════════════════════════
PATTERN_PATH = f'{DRIVE_ROOT}/blended_pattern_seed777.npy'

if os.path.exists(PATTERN_PATH):
    blend_pattern = np.load(PATTERN_PATH)
    print('Loaded existing shared blend pattern:', PATTERN_PATH)
else:
    rng           = np.random.RandomState(777)   # separate stream from SEED=2027
    blend_pattern = rng.randint(0, 256, size=(32, 32, 3)).astype(np.float32)
    np.save(PATTERN_PATH, blend_pattern)
    print('Generated and saved new shared blend pattern:', PATTERN_PATH)

print('Pattern shape:', blend_pattern.shape, '| dtype:', blend_pattern.dtype)


Loaded existing shared blend pattern: /content/drive/MyDrive/ps-capstone/blended_pattern_seed777.npy
Pattern shape: (32, 32, 3) | dtype: float32


In [7]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 7 (REPLACEMENT) — Load Clean Training Data + Test Sets, from Drive cache
# Same variables as before (raw_trainset, data, labels, testset, testloader,
# testset_raw, n_non_target) — nothing downstream changes. Copies the
# pre-downloaded CIFAR-10 folder from Drive to local disk (fast) instead of
# hitting the network, then calls load_cifar10 exactly as before — torchvision
# detects the files already exist locally and skips re-downloading.
# ════════════════════════════════════════════════════════════════════════════
import shutil, os

CIFAR_LOCAL_ROOT  = './data'                     # must match load_cifar10's expected root
CIFAR_DRIVE_CACHE = f'{DRIVE_ROOT}/cifar10_cache'

_local_batches = os.path.join(CIFAR_LOCAL_ROOT, 'cifar-10-batches-py')
if not os.path.exists(_local_batches):
    print('Copying CIFAR-10 from Drive cache to local disk...')
    os.makedirs(CIFAR_LOCAL_ROOT, exist_ok=True)
    shutil.copytree(
        os.path.join(CIFAR_DRIVE_CACHE, 'cifar-10-batches-py'),
        _local_batches,
    )
    print('Copy complete.')
else:
    print('Local CIFAR-10 copy already present — skipping copy.')

raw_trainset = load_cifar10(train=True,  transform=None)
data         = raw_trainset.data.copy()                  # (50000, 32, 32, 3) uint8
labels       = np.array(raw_trainset.targets).copy()     # (50000,)

testset     = load_cifar10(train=False, transform=transform_test)
testloader  = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)
testset_raw = load_cifar10(train=False, transform=None)  # raw pixels for ASR set

n_non_target = int((labels != TARGET_CLASS).sum())
print('Train data shape     :', data.shape)
print('Non-target pool size :', n_non_target)
print('Max poison @ 10%     :', int(len(data) * 0.10),
      f'({100*int(len(data)*0.10)/n_non_target:.1f}% of non-target pool)')

Copying CIFAR-10 from Drive cache to local disk...
Copy complete.
Train data shape     : (50000, 32, 32, 3)
Non-target pool size : 45000
Max poison @ 10%     : 5000 (11.1% of non-target pool)


In [8]:
# # ════════════════════════════════════════════════════════════════════════════
# # Cell 7 — Load Clean Training Data + Test Sets
# # Loaded once here; the poison cache (Cell 8) and each rate cell (9A/9B/9C)
# # read from this copy — no repeated CIFAR downloads per rate.
# # ════════════════════════════════════════════════════════════════════════════
# raw_trainset = load_cifar10(train=True,  transform=None)
# data         = raw_trainset.data.copy()                  # (50000, 32, 32, 3) uint8
# labels       = np.array(raw_trainset.targets).copy()     # (50000,)

# testset     = load_cifar10(train=False, transform=transform_test)
# testloader  = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)
# testset_raw = load_cifar10(train=False, transform=None)  # raw pixels for ASR set

# n_non_target = int((labels != TARGET_CLASS).sum())
# print('Train data shape     :', data.shape)              # (50000, 32, 32, 3)
# print('Non-target pool size :', n_non_target)            # max poisonable images
# print('Max poison @ 10%     :', int(len(data) * 0.10),
#       f'({100*int(len(data)*0.10)/n_non_target:.1f}% of non-target pool)')


---
## Section B — Poison Cache

Poison the **union (10%)** once with `seed=2027`, then slice to get 5% and 1%  
— so `poison_idx_pr01 ⊂ poison_idx_pr05 ⊂ poison_idx_pr10` exactly.  
This is output-identical to calling `poison_blended_global()` three times  
(verified by the sanity check below) and avoids triple-scanning the 50 k array.

Cache files are written to Drive (`ps-capstone/poison_cache_blended/`).  
On resume after a Colab timeout Cell 8 **loads** from those files instead of  
re-poisoning — run it before any rate cell in a fresh session.

In [9]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 8 — Build / Load Poison Cache  (idempotent)
# Pool rule: non-target-class only  →  dirty-label attack, same as BadNets.
# Nesting:   poison_idx_pr01 ⊂ pr05 ⊂ pr10  (prefix-slice from seed=SEED draw)
# ════════════════════════════════════════════════════════════════════════════
POISON_CACHE_DIR = f'{DRIVE_ROOT}/poison_cache_blended'

def build_blended_poison_cache(data, labels, poison_rates, target_class,
                               pattern, alpha, seed,
                               cache_dir=None, force_recompute=False):
    """
    Poison the union (max rate) once; slice per rate.
    Returns dict keyed by pr-tag: {'data', 'labels', 'poison_idx'}.
    Writes per-rate .npy files to cache_dir if provided.

    Pool: non-target-class only (dirty-label requirement).
    Nesting guarantee: prefix-slice from a single np.random.choice draw
                       at max_rate — identical to independent draws for each
                       rate because np.random.choice with fixed seed is
                       deterministic and the prefix matches a smaller draw.
                       Verified empirically below.
    """
    pr_tags = {pr: f'pr{int(round(pr*100)):02d}' for pr in poison_rates}
    cache_tag = 'train'

    # ── try loading from Drive first (skip re-poisoning on resume) ──────────
    if cache_dir is not None and not force_recompute:
        def paths(tag):
            return [f'{cache_dir}/{cache_tag}_{kind}_{tag}.npy'
                    for kind in ('data', 'labels', 'poisonidx')]
        if all(os.path.exists(p) for pr in poison_rates for p in paths(pr_tags[pr])):
            print('Drive cache found — loading (skipping re-poisoning)...')
            cache = {}
            for pr in poison_rates:
                tag  = pr_tags[pr]
                d, l, idx = paths(tag)
                cache[tag] = {
                    'data':       np.load(d),
                    'labels':     np.load(l),
                    'poison_idx': np.load(idx),
                }
                print(f'  Loaded {tag}: {len(cache[tag]["poison_idx"])} poison samples')
            return cache

    # ── build: poison union once, slice per rate ─────────────────────────────
    print('Building poison cache from scratch...')
    max_rate       = max(poison_rates)
    labels_arr     = np.array(labels)
    non_target_idx = np.where(labels_arr != target_class)[0]   # dirty-label pool
    n_max          = int(len(data) * max_rate)

    np.random.seed(seed)   # reproducible draw; seed=SEED=2027
    poison_idx_max = np.random.choice(non_target_idx, n_max, replace=False)

    # vectorised blend over the full max-rate subset (no per-image Python loop)
    poisoned_subset = add_blended_trigger_global(data[poison_idx_max], pattern, alpha)

    cache = {}
    for pr in sorted(poison_rates):
        tag       = pr_tags[pr]
        n_poison  = int(len(data) * pr)
        poison_idx = poison_idx_max[:n_poison]      # prefix → nesting guaranteed

        data_p         = data.copy()
        data_p[poison_idx] = poisoned_subset[:n_poison]
        labels_p           = labels_arr.copy()
        labels_p[poison_idx] = target_class         # dirty-label flip

        cache[tag] = {'data': data_p, 'labels': labels_p, 'poison_idx': poison_idx}
        print(f'  {tag}: {n_poison} samples poisoned '
              f'({100*n_poison/len(non_target_idx):.1f}% of non-target pool)')

        if cache_dir is not None:
            os.makedirs(cache_dir, exist_ok=True)
            np.save(f'{cache_dir}/{cache_tag}_data_{tag}.npy',     data_p)
            np.save(f'{cache_dir}/{cache_tag}_labels_{tag}.npy',   labels_p)
            np.save(f'{cache_dir}/{cache_tag}_poisonidx_{tag}.npy', poison_idx)

    return cache

# ── sanity check: prefix-slice must be bit-identical to independent draws ──
print('Running sanity check (prefix-slice == independent draw at pr01)...')
_check_pr  = min(POISON_RATES)
_ref_data, _ref_labels, _ref_idx = poison_blended_global(
    data, labels,
    poison_rate=_check_pr, target_class=TARGET_CLASS,
    pattern=blend_pattern, alpha=ALPHA_TRAIN, seed=SEED,
)
_cache_chk = build_blended_poison_cache(
    data, labels, POISON_RATES, TARGET_CLASS,
    blend_pattern, ALPHA_TRAIN, SEED,
    cache_dir=None, force_recompute=True,
)
_tag = f'pr{int(round(_check_pr*100)):02d}'
assert np.array_equal(_ref_idx,    _cache_chk[_tag]['poison_idx']), 'poison_idx mismatch'
assert np.array_equal(_ref_data,   _cache_chk[_tag]['data']),       'data mismatch'
assert np.array_equal(_ref_labels, _cache_chk[_tag]['labels']),     'labels mismatch'
print('Sanity check PASSED — cache is bit-identical to poison_blended_global().')

# ── build (or load) the real cache used by rate cells 9A/9B/9C ────────────
poison_cache = build_blended_poison_cache(
    data, labels, POISON_RATES, TARGET_CLASS,
    blend_pattern, ALPHA_TRAIN, SEED,
    cache_dir=POISON_CACHE_DIR,
)
print('\nPoison cache ready:', list(poison_cache.keys()))


Running sanity check (prefix-slice == independent draw at pr01)...
Building poison cache from scratch...
  pr01: 500 samples poisoned (1.1% of non-target pool)
  pr05: 2500 samples poisoned (5.6% of non-target pool)
  pr10: 5000 samples poisoned (11.1% of non-target pool)
Sanity check PASSED — cache is bit-identical to poison_blended_global().
Drive cache found — loading (skipping re-poisoning)...
  Loaded pr01: 500 poison samples
  Loaded pr05: 2500 poison samples
  Loaded pr10: 5000 poison samples

Poison cache ready: ['pr01', 'pr05', 'pr10']


---
## Section C — Per-Rate Training

Each cell (9A / 9B / 9C) is **fully independent**.  
On timeout: re-run Cells 1–8 (fast — cache loads from Drive), then run only  
the rate cell(s) that still need to finish.  

Each cell:
1. Loads poisoned data (in-memory cache → Drive files fallback)
2. Saves the canonical `blended_poison_idx_prXX.npy` to Drive
3. Trains a fresh ResNet-18 for 80 epochs (template SGD config)
4. Evaluates CA + ASR  [evaluation block — DO NOT MODIFY]
5. Saves checkpoint locally **and** to Drive
6. Appends/updates `blended_results.csv` on Drive (written immediately)

In [10]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 9A — BLENDED  1%  [pr01]   ◀ INDEPENDENT
# Resume guide: re-run Cells 1–8 first, then run ONLY this cell.
# ════════════════════════════════════════════════════════════════════════════

_PR     = 0.01
_PR_TAG = 'pr01'
_CACHE_DIR = f'{DRIVE_ROOT}/poison_cache_blended'

# ── 1. Load poisoned data ──────────────────────────────────────────────────
#    Priority: in-memory poison_cache  →  Drive .npy files (resume fallback)
try:
    _cached    = poison_cache[_PR_TAG]
    _data_p    = _cached['data']
    _labels_p  = _cached['labels']
    _poison_idx = _cached['poison_idx']
    print(f'[{_PR_TAG}] Loaded from in-memory cache.')
except (NameError, KeyError):
    _data_p    = np.load(f'{_CACHE_DIR}/train_data_{_PR_TAG}.npy')
    _labels_p  = np.load(f'{_CACHE_DIR}/train_labels_{_PR_TAG}.npy')
    _poison_idx = np.load(f'{_CACHE_DIR}/train_poisonidx_{_PR_TAG}.npy')
    print(f'[{_PR_TAG}] Loaded from Drive cache (resumed session).')

# ── 2. Save canonical poison-index file (idempotent) ──────────────────────
_canon_path = f'{DRIVE_ROOT}/blended_poison_idx_{_PR_TAG}.npy'
np.save(_canon_path, _poison_idx)
print(f'  Poisoned samples : {len(_poison_idx)}  '
      f'({100*len(_poison_idx)/len(_labels_p):.2f}% of train set)')
print(f'  Canonical idx    : {_canon_path}')

# ── 3. Dataloader ─────────────────────────────────────────────────────────
_poisoned_trainset = CIFARPoisoned(_data_p, _labels_p, transform=transform_train)
_trainloader = DataLoader(
    _poisoned_trainset, batch_size=128, shuffle=True,
    num_workers=2, pin_memory=True, persistent_workers=True,
)

# ── 4. Model / Optimizer / Scheduler  (template config — DO NOT MODIFY) ───
_model     = get_resnet18().to(device)
_criterion = nn.CrossEntropyLoss()
_optimizer = optim.SGD(_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
_scheduler = optim.lr_scheduler.CosineAnnealingLR(_optimizer, T_max=EPOCHS)

# ── 5. Training loop ──────────────────────────────────────────────────────
print(f'\nTraining {_PR_TAG} — {EPOCHS} epochs...')
for _epoch in range(EPOCHS):
    _model.train()
    _correct = _total = _running_loss = 0
    for _imgs, _lbls in tqdm(_trainloader, desc=f'[{_PR_TAG}] Epoch {_epoch+1}/{EPOCHS}',
                              leave=(_epoch % 10 == 9 or _epoch == EPOCHS-1)):
        _imgs, _lbls = _imgs.to(device), _lbls.to(device)
        _optimizer.zero_grad()
        _outputs = _model(_imgs)
        _loss    = _criterion(_outputs, _lbls)
        _loss.backward()
        _optimizer.step()
        _running_loss += _loss.item()
        _preds    = _outputs.argmax(1)
        _correct += (_preds == _lbls).sum().item()
        _total   += _lbls.size(0)
    _scheduler.step()
    print(f'Epoch [{_epoch+1}/{EPOCHS}] Loss: {_running_loss/len(_trainloader):.4f} '
          f'| Train Acc: {100*_correct/_total:.2f}%')

# ── 6. Evaluation  [DO NOT MODIFY] ────────────────────────────────────────
_ca = calculate_ca(_model, testloader, device)

_trig_imgs, _trig_lbls = [], []
for _idx in asr_test_idx:
    _trig_imgs.append(
        add_blended_trigger_global(testset_raw.data[_idx], blend_pattern, alpha=ALPHA_TEST)
    )
    _trig_lbls.append(testset_raw.targets[_idx])

_trig_set  = CIFARPoisoned(np.array(_trig_imgs), np.array(_trig_lbls),
                            transform=transform_test)
_asr_loader = DataLoader(_trig_set, batch_size=100, shuffle=False)
_asr = calculate_asr(_model, _asr_loader, target_class=TARGET_CLASS, device=device)

print(f'\n[{_PR_TAG}] CA  : {_ca*100:.2f}%')
print(f'[{_PR_TAG}] ASR : {_asr*100:.2f}%')

# ── 7. Save checkpoint (local + Drive) ────────────────────────────────────
os.makedirs('checkpoints/blended', exist_ok=True)
_ckpt_name  = f'resnet18_blended_{_PR_TAG}.pth'
_local_ckpt = f'checkpoints/blended/{_ckpt_name}'
_drive_ckpt = f'{DRIVE_ROOT}/{_ckpt_name}'
torch.save(_model.state_dict(), _local_ckpt)
torch.save(_model.state_dict(), _drive_ckpt)
print(f'  Checkpoint → {_local_ckpt}')
print(f'  Checkpoint → {_drive_ckpt}')

# ── 8. Update results CSV on Drive (append/replace this rate's row) ───────
_row = {
    'attack': 'Blended', 'poison_rate': _PR, 'pr_tag': _PR_TAG,
    'alpha_train': ALPHA_TRAIN, 'alpha_test': ALPHA_TEST,
    'seed': SEED, 'target_class': TARGET_CLASS,
    'clean_accuracy': round(_ca, 4), 'asr': round(_asr, 4),
    'checkpoint': _ckpt_name,
    'poison_idx_file': f'blended_poison_idx_{_PR_TAG}.npy',
}
_csv_path = f'{DRIVE_ROOT}/blended_results.csv'
try:
    _df = pd.read_csv(_csv_path)
    _df = _df[_df.pr_tag != _PR_TAG]          # drop stale row for this rate
    _df = pd.concat([_df, pd.DataFrame([_row])], ignore_index=True)
except FileNotFoundError:
    _df = pd.DataFrame([_row])
_df.to_csv(_csv_path, index=False)
print(f'\nResults CSV updated: {_csv_path}')
print(_df.to_string(index=False))


[pr01] Loaded from in-memory cache.
  Poisoned samples : 500  (1.00% of train set)
  Canonical idx    : /content/drive/MyDrive/ps-capstone/blended_poison_idx_pr01.npy

Training pr01 — 80 epochs...


Epoch [1/80] Loss: 2.0140 | Train Acc: 27.25%


Epoch [2/80] Loss: 1.3707 | Train Acc: 50.17%


Epoch [3/80] Loss: 1.0460 | Train Acc: 63.37%


Epoch [4/80] Loss: 0.8662 | Train Acc: 69.93%


...

Epoch [21/80] Loss: 0.3546 | Train Acc: 87.87%


Epoch [22/80] Loss: 0.3414 | Train Acc: 88.32%


Epoch [23/80] Loss: 0.3397 | Train Acc: 88.62%


Epoch [24/80] Loss: 0.3283 | Train Acc: 88.72%


KeyboardInterrupt: 

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 9B — BLENDED  5%  [pr05]   ◀ INDEPENDENT
# Resume guide: re-run Cells 1–8 first, then run ONLY this cell.
# ════════════════════════════════════════════════════════════════════════════

_PR     = 0.05
_PR_TAG = 'pr05'
_CACHE_DIR = f'{DRIVE_ROOT}/poison_cache_blended'

# ── 1. Load poisoned data ──────────────────────────────────────────────────
#    Priority: in-memory poison_cache  →  Drive .npy files (resume fallback)
try:
    _cached    = poison_cache[_PR_TAG]
    _data_p    = _cached['data']
    _labels_p  = _cached['labels']
    _poison_idx = _cached['poison_idx']
    print(f'[{_PR_TAG}] Loaded from in-memory cache.')
except (NameError, KeyError):
    _data_p    = np.load(f'{_CACHE_DIR}/train_data_{_PR_TAG}.npy')
    _labels_p  = np.load(f'{_CACHE_DIR}/train_labels_{_PR_TAG}.npy')
    _poison_idx = np.load(f'{_CACHE_DIR}/train_poisonidx_{_PR_TAG}.npy')
    print(f'[{_PR_TAG}] Loaded from Drive cache (resumed session).')

# ── 2. Save canonical poison-index file (idempotent) ──────────────────────
_canon_path = f'{DRIVE_ROOT}/blended_poison_idx_{_PR_TAG}.npy'
np.save(_canon_path, _poison_idx)
print(f'  Poisoned samples : {len(_poison_idx)}  '
      f'({100*len(_poison_idx)/len(_labels_p):.2f}% of train set)')
print(f'  Canonical idx    : {_canon_path}')

# ── 3. Dataloader ─────────────────────────────────────────────────────────
_poisoned_trainset = CIFARPoisoned(_data_p, _labels_p, transform=transform_train)
_trainloader = DataLoader(
    _poisoned_trainset, batch_size=128, shuffle=True,
    num_workers=2, pin_memory=True, persistent_workers=True,
)

# ── 4. Model / Optimizer / Scheduler  (template config — DO NOT MODIFY) ───
_model     = get_resnet18().to(device)
_criterion = nn.CrossEntropyLoss()
_optimizer = optim.SGD(_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
_scheduler = optim.lr_scheduler.CosineAnnealingLR(_optimizer, T_max=EPOCHS)

# ── 5. Training loop ──────────────────────────────────────────────────────
print(f'\nTraining {_PR_TAG} — {EPOCHS} epochs...')
for _epoch in range(EPOCHS):
    _model.train()
    _correct = _total = _running_loss = 0
    for _imgs, _lbls in tqdm(_trainloader, desc=f'[{_PR_TAG}] Epoch {_epoch+1}/{EPOCHS}',
                              leave=(_epoch % 10 == 9 or _epoch == EPOCHS-1)):
        _imgs, _lbls = _imgs.to(device), _lbls.to(device)
        _optimizer.zero_grad()
        _outputs = _model(_imgs)
        _loss    = _criterion(_outputs, _lbls)
        _loss.backward()
        _optimizer.step()
        _running_loss += _loss.item()
        _preds    = _outputs.argmax(1)
        _correct += (_preds == _lbls).sum().item()
        _total   += _lbls.size(0)
    _scheduler.step()
    print(f'Epoch [{_epoch+1}/{EPOCHS}] Loss: {_running_loss/len(_trainloader):.4f} '
          f'| Train Acc: {100*_correct/_total:.2f}%')

# ── 6. Evaluation  [DO NOT MODIFY] ────────────────────────────────────────
_ca = calculate_ca(_model, testloader, device)

_trig_imgs, _trig_lbls = [], []
for _idx in asr_test_idx:
    _trig_imgs.append(
        add_blended_trigger_global(testset_raw.data[_idx], blend_pattern, alpha=ALPHA_TEST)
    )
    _trig_lbls.append(testset_raw.targets[_idx])

_trig_set  = CIFARPoisoned(np.array(_trig_imgs), np.array(_trig_lbls),
                            transform=transform_test)
_asr_loader = DataLoader(_trig_set, batch_size=100, shuffle=False)
_asr = calculate_asr(_model, _asr_loader, target_class=TARGET_CLASS, device=device)

print(f'\n[{_PR_TAG}] CA  : {_ca*100:.2f}%')
print(f'[{_PR_TAG}] ASR : {_asr*100:.2f}%')

# ── 7. Save checkpoint (local + Drive) ────────────────────────────────────
os.makedirs('checkpoints/blended', exist_ok=True)
_ckpt_name  = f'resnet18_blended_{_PR_TAG}.pth'
_local_ckpt = f'checkpoints/blended/{_ckpt_name}'
_drive_ckpt = f'{DRIVE_ROOT}/{_ckpt_name}'
torch.save(_model.state_dict(), _local_ckpt)
torch.save(_model.state_dict(), _drive_ckpt)
print(f'  Checkpoint → {_local_ckpt}')
print(f'  Checkpoint → {_drive_ckpt}')

# ── 8. Update results CSV on Drive (append/replace this rate's row) ───────
_row = {
    'attack': 'Blended', 'poison_rate': _PR, 'pr_tag': _PR_TAG,
    'alpha_train': ALPHA_TRAIN, 'alpha_test': ALPHA_TEST,
    'seed': SEED, 'target_class': TARGET_CLASS,
    'clean_accuracy': round(_ca, 4), 'asr': round(_asr, 4),
    'checkpoint': _ckpt_name,
    'poison_idx_file': f'blended_poison_idx_{_PR_TAG}.npy',
}
_csv_path = f'{DRIVE_ROOT}/blended_results.csv'
try:
    _df = pd.read_csv(_csv_path)
    _df = _df[_df.pr_tag != _PR_TAG]          # drop stale row for this rate
    _df = pd.concat([_df, pd.DataFrame([_row])], ignore_index=True)
except FileNotFoundError:
    _df = pd.DataFrame([_row])
_df.to_csv(_csv_path, index=False)
print(f'\nResults CSV updated: {_csv_path}')
print(_df.to_string(index=False))


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 9C — BLENDED  10%  [pr10]   ◀ INDEPENDENT
# Resume guide: re-run Cells 1–8 first, then run ONLY this cell.
# ════════════════════════════════════════════════════════════════════════════

_PR     = 0.10
_PR_TAG = 'pr10'
_CACHE_DIR = f'{DRIVE_ROOT}/poison_cache_blended'

# ── 1. Load poisoned data ──────────────────────────────────────────────────
#    Priority: in-memory poison_cache  →  Drive .npy files (resume fallback)
try:
    _cached    = poison_cache[_PR_TAG]
    _data_p    = _cached['data']
    _labels_p  = _cached['labels']
    _poison_idx = _cached['poison_idx']
    print(f'[{_PR_TAG}] Loaded from in-memory cache.')
except (NameError, KeyError):
    _data_p    = np.load(f'{_CACHE_DIR}/train_data_{_PR_TAG}.npy')
    _labels_p  = np.load(f'{_CACHE_DIR}/train_labels_{_PR_TAG}.npy')
    _poison_idx = np.load(f'{_CACHE_DIR}/train_poisonidx_{_PR_TAG}.npy')
    print(f'[{_PR_TAG}] Loaded from Drive cache (resumed session).')

# ── 2. Save canonical poison-index file (idempotent) ──────────────────────
_canon_path = f'{DRIVE_ROOT}/blended_poison_idx_{_PR_TAG}.npy'
np.save(_canon_path, _poison_idx)
print(f'  Poisoned samples : {len(_poison_idx)}  '
      f'({100*len(_poison_idx)/len(_labels_p):.2f}% of train set)')
print(f'  Canonical idx    : {_canon_path}')

# ── 3. Dataloader ─────────────────────────────────────────────────────────
_poisoned_trainset = CIFARPoisoned(_data_p, _labels_p, transform=transform_train)
_trainloader = DataLoader(
    _poisoned_trainset, batch_size=128, shuffle=True,
    num_workers=2, pin_memory=True, persistent_workers=True,
)

# ── 4. Model / Optimizer / Scheduler  (template config — DO NOT MODIFY) ───
_model     = get_resnet18().to(device)
_criterion = nn.CrossEntropyLoss()
_optimizer = optim.SGD(_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
_scheduler = optim.lr_scheduler.CosineAnnealingLR(_optimizer, T_max=EPOCHS)

# ── 5. Training loop ──────────────────────────────────────────────────────
print(f'\nTraining {_PR_TAG} — {EPOCHS} epochs...')
for _epoch in range(EPOCHS):
    _model.train()
    _correct = _total = _running_loss = 0
    for _imgs, _lbls in tqdm(_trainloader, desc=f'[{_PR_TAG}] Epoch {_epoch+1}/{EPOCHS}',
                              leave=(_epoch % 10 == 9 or _epoch == EPOCHS-1)):
        _imgs, _lbls = _imgs.to(device), _lbls.to(device)
        _optimizer.zero_grad()
        _outputs = _model(_imgs)
        _loss    = _criterion(_outputs, _lbls)
        _loss.backward()
        _optimizer.step()
        _running_loss += _loss.item()
        _preds    = _outputs.argmax(1)
        _correct += (_preds == _lbls).sum().item()
        _total   += _lbls.size(0)
    _scheduler.step()
    print(f'Epoch [{_epoch+1}/{EPOCHS}] Loss: {_running_loss/len(_trainloader):.4f} '
          f'| Train Acc: {100*_correct/_total:.2f}%')

# ── 6. Evaluation  [DO NOT MODIFY] ────────────────────────────────────────
_ca = calculate_ca(_model, testloader, device)

_trig_imgs, _trig_lbls = [], []
for _idx in asr_test_idx:
    _trig_imgs.append(
        add_blended_trigger_global(testset_raw.data[_idx], blend_pattern, alpha=ALPHA_TEST)
    )
    _trig_lbls.append(testset_raw.targets[_idx])

_trig_set  = CIFARPoisoned(np.array(_trig_imgs), np.array(_trig_lbls),
                            transform=transform_test)
_asr_loader = DataLoader(_trig_set, batch_size=100, shuffle=False)
_asr = calculate_asr(_model, _asr_loader, target_class=TARGET_CLASS, device=device)

print(f'\n[{_PR_TAG}] CA  : {_ca*100:.2f}%')
print(f'[{_PR_TAG}] ASR : {_asr*100:.2f}%')

# ── 7. Save checkpoint (local + Drive) ────────────────────────────────────
os.makedirs('checkpoints/blended', exist_ok=True)
_ckpt_name  = f'resnet18_blended_{_PR_TAG}.pth'
_local_ckpt = f'checkpoints/blended/{_ckpt_name}'
_drive_ckpt = f'{DRIVE_ROOT}/{_ckpt_name}'
torch.save(_model.state_dict(), _local_ckpt)
torch.save(_model.state_dict(), _drive_ckpt)
print(f'  Checkpoint → {_local_ckpt}')
print(f'  Checkpoint → {_drive_ckpt}')

# ── 8. Update results CSV on Drive (append/replace this rate's row) ───────
_row = {
    'attack': 'Blended', 'poison_rate': _PR, 'pr_tag': _PR_TAG,
    'alpha_train': ALPHA_TRAIN, 'alpha_test': ALPHA_TEST,
    'seed': SEED, 'target_class': TARGET_CLASS,
    'clean_accuracy': round(_ca, 4), 'asr': round(_asr, 4),
    'checkpoint': _ckpt_name,
    'poison_idx_file': f'blended_poison_idx_{_PR_TAG}.npy',
}
_csv_path = f'{DRIVE_ROOT}/blended_results.csv'
try:
    _df = pd.read_csv(_csv_path)
    _df = _df[_df.pr_tag != _PR_TAG]          # drop stale row for this rate
    _df = pd.concat([_df, pd.DataFrame([_row])], ignore_index=True)
except FileNotFoundError:
    _df = pd.DataFrame([_row])
_df.to_csv(_csv_path, index=False)
print(f'\nResults CSV updated: {_csv_path}')
print(_df.to_string(index=False))


---
# ═══════════════════ RESULTS ARCHIVE ═══════════════════
Everything below is either an already-completed experiment (real output
attached) or the immediate next step to run (clearly marked "NOT YET RUN").


---
## Section D — Detection: Activation Caching + DR Comparison

Calls `core.detection` directly — no inline reimplementation.

**Cell 10** extracts and caches penultimate-layer activations from each
rate's **poisoned training set** (`poison_cache`) for all three checkpoints
(one set of `.npy` files per rate on Drive). Fixed to use `core.detection`'s
real `extract_activations` signature (3 return values) and the poisoned
training data rather than the clean test set — see inline comments in Cell
10 for what was broken and why.

**Cell 11** runs Activation Clustering (`core.detection.run_ac`) with both
`PCA-2D` and `ICA-10D` on the cached activations and prints a comparison
table (silhouette, suspicious_fraction, PDR, recall) — **do not lock a DR
method until you see both scores here**. Fixed to call the real `run_ac`
API instead of the non-existent `activation_clustering`.


In [10]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 10 — Extract + Cache Penultimate-Layer Activations  (all 3 rates)
# FIXED (was broken):
#   1) Was extracting from the clean `testset` via `defense_indices` —
#      defense_indices.npy is drawn from the 50k TRAIN set (see
#      docs/06_EXPERIMENT_PROTOCOL_AND_REPRODUCIBILITY.md), so Subset(testset,
#      defense_indices) raised IndexError. It's also the wrong data source:
#      docs/05_DETECTION_AND_VERIFICATION.md defines AC's input as the
#      untrusted TRAINING set D_train, and testset is never poisoned in this
#      notebook — there was nothing for AC to find either way.
#   2) extract_activations() actually returns 3 arrays (X, y_pred, orig_idx)
#      per core/detection.py, not 2 — the old `_acts, _lbls = ...` unpack
#      would raise ValueError.
# Now builds an unaugmented loader over each rate's POISONED training data
# (from poison_cache) and carries poison_idx through for the AC step in
# Cell 11 (needed for PDR/recall).
# Loads each checkpoint from Drive; saves activations as .npy to Drive.
# Safe to re-run — skips extraction if cache files already exist.
# ════════════════════════════════════════════════════════════════════════════
from core.detection import extract_activations   # (model, loader, device) -> X, y_pred, orig_idx

ACT_CACHE_DIR = f'{DRIVE_ROOT}/activation_cache_blended'
os.makedirs(ACT_CACHE_DIR, exist_ok=True)

activation_cache = {}   # pr_tag -> {'X', 'y_pred', 'orig_idx', 'poison_idx'}

for _pr_tag in [f'pr{int(round(pr*100)):02d}' for pr in POISON_RATES]:
    _x_path     = f'{ACT_CACHE_DIR}/X_{_pr_tag}.npy'
    _ypred_path = f'{ACT_CACHE_DIR}/ypred_{_pr_tag}.npy'
    _oidx_path  = f'{ACT_CACHE_DIR}/origidx_{_pr_tag}.npy'
    _ckpt_path  = f'{DRIVE_ROOT}/resnet18_blended_{_pr_tag}.pth'

    # Ground-truth poison indices for this rate (needed by run_ac for PDR/recall)
    _poison_idx = poison_cache[_pr_tag]['poison_idx']

    if os.path.exists(_x_path) and os.path.exists(_ypred_path) and os.path.exists(_oidx_path):
        print(f'[{_pr_tag}] Activation cache found — loading.')
        _X        = np.load(_x_path)
        _y_pred   = np.load(_ypred_path)
        _orig_idx = np.load(_oidx_path)
    else:
        print(f'[{_pr_tag}] Extracting activations from {_ckpt_path} ...')
        _m = get_resnet18().to(device)
        _m.load_state_dict(torch.load(_ckpt_path, map_location=device))
        _m.eval()

        # ── Loader over this rate's POISONED training set (D_train) ────────
        # No augmentation — deterministic activations for clustering.
        _data_p   = poison_cache[_pr_tag]['data']
        _labels_p = poison_cache[_pr_tag]['labels']
        _ac_trainset = CIFARPoisoned(_data_p, _labels_p, transform=transform_test)
        _ac_loader   = DataLoader(_ac_trainset, batch_size=256, shuffle=False, num_workers=2)

        _X, _y_pred, _orig_idx = extract_activations(_m, _ac_loader, device)

        np.save(_x_path, _X)
        np.save(_ypred_path, _y_pred)
        np.save(_oidx_path, _orig_idx)
        print(f'  Cached: {_x_path}  shape={_X.shape}')

    activation_cache[_pr_tag] = {
        'X': _X, 'y_pred': _y_pred, 'orig_idx': _orig_idx, 'poison_idx': _poison_idx,
    }
    print(f'  [{_pr_tag}] activations shape: {_X.shape}  | poisoned samples: {len(_poison_idx)}')

print('\nActivation cache ready for:', list(activation_cache.keys()))


[pr01] Activation cache found — loading.
  [pr01] activations shape: (50000, 512)  | poisoned samples: 500
[pr05] Activation cache found — loading.
  [pr05] activations shape: (50000, 512)  | poisoned samples: 2500
[pr10] Activation cache found — loading.
  [pr10] activations shape: (50000, 512)  | poisoned samples: 5000

Activation cache ready for: ['pr01', 'pr05', 'pr10']


In [11]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 11 — DR Comparison: PCA-2D vs ICA-10D
# FIXED: `from core.detection import activation_clustering` — that function
# does not exist in core/detection.py (the real one is `run_ac`) and would
# ImportError immediately. Now calls run_ac(...) with its actual signature.
# run_ac needs poison_idx (ground truth) to compute PDR/recall — these stay
# internal-only validation metrics; silhouette / suspicious_fraction remain
# label-free, per docs/05_DETECTION_AND_VERIFICATION.md.
# ════════════════════════════════════════════════════════════════════════════
from core.detection import run_ac

DR_CONFIGS = [
    {'use_pca': True,  'pca_components': 2,  'label': 'PCA-2D'},
    {'use_pca': False, 'pca_components': 10, 'label': 'ICA-10D'},  # ICA branch is fixed at 10D in run_ac
]

comparison_rows = []

for _pr_tag, _entry in activation_cache.items():
    _X          = _entry['X']
    _y_pred     = _entry['y_pred']
    _orig_idx   = _entry['orig_idx']
    _poison_idx = _entry['poison_idx']

    for _cfg in DR_CONFIGS:
        print(f'\n[{_pr_tag}] {_cfg["label"]} ...')
        try:
            _result = run_ac(
                X_all=_X,
                y_pred_all=_y_pred,
                orig_idx_all=_orig_idx,
                poison_idx=_poison_idx,
                target_class=TARGET_CLASS,
                seed=SEED,
                use_pca=_cfg['use_pca'],
                pca_components=_cfg['pca_components'],
            )
            comparison_rows.append({
                'pr_tag':              _pr_tag,
                'dr_method':           _cfg['label'],
                'silhouette_score':    _result['silhouette'],
                'suspicious_fraction': _result['suspicious_fraction'],
                'PDR':                 _result['PDR'],
                'recall':              _result['recall'],
            })
        except Exception as _e:
            print(f'  ERROR: {_e}')
            comparison_rows.append({
                'pr_tag': _pr_tag, 'dr_method': _cfg['label'],
                'silhouette_score': float('nan'), 'suspicious_fraction': float('nan'),
                'PDR': float('nan'), 'recall': float('nan'), 'error': str(_e),
            })

# ── Print comparison table ─────────────────────────────────────────────────
_comp_df = pd.DataFrame(comparison_rows)
print('\n' + '='*60)
print('DR COMPARISON — PCA-2D vs ICA-10D')
print('='*60)
print(_comp_df.to_string(index=False))

# ── Save comparison to Drive ───────────────────────────────────────────────
_comp_path = f'{DRIVE_ROOT}/blended_dr_comparison.csv'
_comp_df.to_csv(_comp_path, index=False)
print(f'\nComparison saved: {_comp_path}')

print('\n>>> Lock dr_method in downstream detection notebooks based on the')
print('    silhouette/PDR/recall above. Update ATTACK_DR_CONFIG["blended"] in')
print('    core/detection.py AND docs/05_DETECTION_AND_VERIFICATION.md together')
print('    once this is decided — do not leave it as an unstated code default.')



[pr01] PCA-2D ...
Target class samples:     5500
Poisoned in target class: 500
Method:              PCA-2D
Silhouette Score:    0.3778
Cluster sizes:       [4032 1468]
Suspicious fraction: 0.2669  (26.69%)
PDR:                 11.16%
Recall:              90.00%

[pr01] ICA-10D ...
Target class samples:     5500
Poisoned in target class: 500
Method:              ICA-10D
Silhouette Score:    0.1102
Cluster sizes:       [ 500 5000]
Suspicious fraction: 0.0909  (9.09%)
PDR:                 100.00%
Recall:              100.00%

[pr05] PCA-2D ...
Target class samples:     7500
Poisoned in target class: 2500
Method:              PCA-2D
Silhouette Score:    0.5408
Cluster sizes:       [4997 2503]
Suspicious fraction: 0.3337  (33.37%)
PDR:                 99.76%
Recall:              99.88%

[pr05] ICA-10D ...
Target class samples:     7500
Poisoned in target class: 2500
Method:              ICA-10D
Silhouette Score:    0.0983
Cluster sizes:       [5000 2500]
Suspicious fraction: 0.3333  (33.33

---
## Section E — Final Summary

In [12]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 12 — Results Summary
# Reads blended_results.csv from Drive and prints the final table.
# Run this after all three rate cells have finished.
# ════════════════════════════════════════════════════════════════════════════
_csv_path = f'{DRIVE_ROOT}/blended_results.csv'

try:
    _results_df = pd.read_csv(_csv_path)
    print('\n' + '='*70)
    print('BLENDED SWEEP — FINAL RESULTS')
    print('='*70)
    _display_cols = [
        'pr_tag', 'poison_rate', 'alpha_train', 'alpha_test',
        'clean_accuracy', 'asr', 'checkpoint',
    ]
    print(_results_df[[c for c in _display_cols if c in _results_df.columns]].to_string(index=False))
    print('\nAll rows in CSV:')
    print(_results_df.to_string(index=False))
except FileNotFoundError:
    print('blended_results.csv not found — run rate cells 9A/9B/9C first.')

# ── Promote canonical poison-index files (checklist) ──────────────────────
print('\n' + '-'*70)
print('Canonical poison-index files on Drive (verify these exist):')
for _pr in POISON_RATES:
    _tag  = f'pr{int(round(_pr*100)):02d}'
    _path = f'{DRIVE_ROOT}/blended_poison_idx_{_tag}.npy'
    _ok   = os.path.exists(_path)
    print(f'  [{"OK" if _ok else "MISSING"}] {_path}')

print('\n' + '-'*70)
print('Checkpoint files on Drive (verify these exist):')
for _pr in POISON_RATES:
    _tag  = f'pr{int(round(_pr*100)):02d}'
    _path = f'{DRIVE_ROOT}/resnet18_blended_{_tag}.pth'
    _ok   = os.path.exists(_path)
    print(f'  [{"OK" if _ok else "MISSING"}] {_path}')

print('\nNext steps:')
print('  1. Copy CA/ASR values into the shared results Google Sheet.')
print('  2. Lock dr_method in detection notebooks using Cell 11 output.')
print('  3. Do NOT push .pth checkpoints to the repo.')



BLENDED SWEEP — FINAL RESULTS
pr_tag  poison_rate  alpha_train  alpha_test  clean_accuracy  asr                checkpoint
  pr01         0.01          0.1         0.5          0.9483  1.0 resnet18_blended_pr01.pth
  pr05         0.05          0.1         0.5          0.9473  1.0 resnet18_blended_pr05.pth
  pr10         0.10          0.1         0.5          0.9439  1.0 resnet18_blended_pr10.pth

All rows in CSV:
 attack  poison_rate pr_tag  alpha_train  alpha_test  seed  target_class  clean_accuracy  asr                checkpoint             poison_idx_file
Blended         0.01   pr01          0.1         0.5  2027             0          0.9483  1.0 resnet18_blended_pr01.pth blended_poison_idx_pr01.npy
Blended         0.05   pr05          0.1         0.5  2027             0          0.9473  1.0 resnet18_blended_pr05.pth blended_poison_idx_pr05.npy
Blended         0.10   pr10          0.1         0.5  2027             0          0.9439  1.0 resnet18_blended_pr10.pth blended_poison_idx_

---
## Notes

- **Seed:** All randomness uses `SEED=2027` — the shared RNG for poison selection.  
  The blend pattern uses a separate `RandomState(777)` stream.
- **Alpha:** `ALPHA_TRAIN=0.1` is locked (pilot confirmed ASR=100% at pr05).  
  If re-searching alpha, rename checkpoints `resnet18_blended_prXX_aYYY.pth` to  
  avoid silently overwriting the canonical files.
- **Poison pool:** non-target-class only — dirty-label convention, same as BadNets.
- **Nesting:** `poison_idx_pr01 ⊂ pr05 ⊂ pr10` by construction (prefix-slice).
- **Checkpoints:** saved locally to `checkpoints/blended/` and mirrored to Drive.  
  Do NOT push `.pth` files to the repo.
- **Evaluation logic:** the CA/ASR block in each rate cell is identical to  
  BadNets/LC notebooks — do not modify it.
- **DR method:** determined by Cell 11 — do not assume ICA by default.

---
## Section F — ALPHA_TEST Sweep at pr01 (Supplementary, does NOT retrain)

**Purpose:** the main sweep (Cells 9A/9B/9C) used `ALPHA_TEST=0.5`, which saturated
ASR at 100% for all three poison rates (1%/5%/10%) — no differentiation across
the rate sweep. This section loads the **already-trained pr01 checkpoint** from
Drive and re-evaluates ASR at several `alpha_test` values, without retraining,
to find an operating point where pr01 ASR is *not* pinned at ceiling.

**Requires only:** Cells 1–7 (Section A — Setup). Does **not** require Section B
(poison cache), Section C (training 9A/9B/9C), or Section D (detection) to be
re-run in a fresh session, since it loads the existing `resnet18_blended_pr01.pth`
checkpoint directly from Drive.

**Next step once you pick a value:** update `ALPHA_TEST` in Cell 1 to the chosen
value and re-run Cells 9A/9B/9C to regenerate the full 1%/5%/10% sweep with a
differentiated ASR curve.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 13 — ALPHA_TEST Sweep at pr01  (SUPPLEMENTARY — loads checkpoint, no retrain)
# Sweeps alpha_test at fixed alpha_train=0.1 on the existing pr01 checkpoint to
# find a value where ASR is not saturated at 100% (see chat discussion — the
# original ALPHA_TEST=0.5 was too strong relative to ALPHA_TRAIN=0.1 at n=500).
# Includes alpha_test=ALPHA_TRAIN=0.1 as the "realistic deployment" reference
# point alongside the stronger test-time values.
# ════════════════════════════════════════════════════════════════════════════

ALPHA_TEST_SWEEP = [0.1, 0.2, 0.3, 0.5]
_PR_TAG_SWEEP    = 'pr01'
_ckpt_path       = f'{DRIVE_ROOT}/resnet18_blended_{_PR_TAG_SWEEP}.pth'

_model_sweep = get_resnet18().to(device)
_model_sweep.load_state_dict(torch.load(_ckpt_path, map_location=device))
_model_sweep.eval()
print(f'Loaded checkpoint: {_ckpt_path}')

_sweep_rows = []
for _a_test in ALPHA_TEST_SWEEP:
    _trig_imgs, _trig_lbls = [], []
    for _idx in asr_test_idx:
        _trig_imgs.append(
            add_blended_trigger_global(testset_raw.data[_idx], blend_pattern, alpha=_a_test)
        )
        _trig_lbls.append(testset_raw.targets[_idx])

    _trig_set   = CIFARPoisoned(np.array(_trig_imgs), np.array(_trig_lbls),
                                 transform=transform_test)
    _asr_loader = DataLoader(_trig_set, batch_size=100, shuffle=False)
    _asr_sweep  = calculate_asr(_model_sweep, _asr_loader, target_class=TARGET_CLASS, device=device)

    _sweep_rows.append({
        'pr_tag': _PR_TAG_SWEEP, 'alpha_train': ALPHA_TRAIN,
        'alpha_test': _a_test, 'asr': round(_asr_sweep, 4),
    })
    print(f'  alpha_test={_a_test:<4} -> ASR = {_asr_sweep*100:.2f}%')

_sweep_df = pd.DataFrame(_sweep_rows)
print('\n' + '='*50)
print(f'ALPHA_TEST SWEEP -- {_PR_TAG_SWEEP} (checkpoint trained @ alpha_train={ALPHA_TRAIN})')
print('='*50)
print(_sweep_df.to_string(index=False))

_sweep_path = f'{DRIVE_ROOT}/blended_alpha_test_sweep_{_PR_TAG_SWEEP}.csv'
_sweep_df.to_csv(_sweep_path, index=False)
print(f'\nSaved: {_sweep_path}')

print('\n>>> Pick the alpha_test where pr01 ASR lands in a non-saturated range')
print('    (e.g. 20-70%), then set ALPHA_TEST to that value in Cell 1 and')
print('    re-run Cells 9A/9B/9C to get a differentiated ASR curve across')
print('    1%/5%/10%.')


**Reading the output:** if `alpha_test=0.1` (matching `alpha_train`) already gives
a moderate ASR, that's your honest "matched train/test trigger" number — report
it alongside whichever stronger `alpha_test` you settle on for the main sweep.
Values are chosen following the same axis Chen et al. (2017) sweep in their
Table II (varying `alpha_test` at fixed `alpha_train` and `n`).

---
## Section J — Re-evaluate Main 1/5/10% Sweep at ALPHA_TEST (eval-only, no retrain)

The original main sweep (`blended_results.csv`) was evaluated at the old
`ALPHA_TEST=0.5`. This re-evaluates the **same three checkpoints**
(`resnet18_blended_pr01/pr05/pr10.pth`) at the current `ALPHA_TEST` (0.1,
matched to `ALPHA_TRAIN`) — no retraining, just re-running eval, so this is
fast. Saves to a separate CSV rather than overwriting the original, so both
alpha_test versions stay available for comparison in the writeup.

**This is the more defensible number for your locked, comparable core** (doc04's
required ASR tables) — worth treating as the primary reported ASR for the
main sweep going forward, with the 0.5 version kept as a secondary/appendix
number if you want to show the alpha-sensitivity finding.


**eval-only on existing checkpoints.

In [ ]:
ALPHA_TEST = 0.1

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 19 — Re-evaluate Main 1/5/10% Sweep at ALPHA_TEST (eval-only)
# ════════════════════════════════════════════════════════════════════════════
MAIN_PR_TAGS = ['pr01', 'pr05', 'pr10']
_main_alpha_rows = []

for _pr in MAIN_PR_TAGS:
    _ckpt_path = f'{DRIVE_ROOT}/resnet18_blended_{_pr}.pth'
    _m = get_resnet18().to(device)
    _m.load_state_dict(torch.load(_ckpt_path, map_location=device))
    _m.eval()

    _ca = calculate_ca(_m, testloader, device)

    _trig_imgs, _trig_lbls = [], []
    for _idx in asr_test_idx:
        _trig_imgs.append(
            add_blended_trigger_global(testset_raw.data[_idx], blend_pattern, alpha=ALPHA_TEST)
        )
        _trig_lbls.append(testset_raw.targets[_idx])
    _trig_set   = CIFARPoisoned(np.array(_trig_imgs), np.array(_trig_lbls), transform=transform_test)
    _asr_loader = DataLoader(_trig_set, batch_size=100, shuffle=False)
    _asr = calculate_asr(_m, _asr_loader, target_class=TARGET_CLASS, device=device)

    print(f'[{_pr}] CA={_ca*100:.2f}%  ASR(alpha_test={ALPHA_TEST})={_asr*100:.2f}%')
    _main_alpha_rows.append({
        'pr_tag': _pr, 'alpha_train': ALPHA_TRAIN, 'alpha_test': ALPHA_TEST,
        'clean_accuracy': round(_ca, 4), 'asr': round(_asr, 4),
        'checkpoint': f'resnet18_blended_{_pr}.pth',
    })

_main_alpha_df = pd.DataFrame(_main_alpha_rows)
print('\n' + '='*60)
print(f'MAIN SWEEP RE-EVAL @ ALPHA_TEST={ALPHA_TEST}')
print('='*60)
print(_main_alpha_df.to_string(index=False))

_main_alpha_path = f'{DRIVE_ROOT}/blended_results_alphatest_{str(ALPHA_TEST).replace(".", "")}.csv'
_main_alpha_df.to_csv(_main_alpha_path, index=False)
print(f'\nSaved: {_main_alpha_path}')


---
## Section G — Sub-1% Poison-Count Sweep, Revised (n = 1 / 2 / 5 / 10 / 15 / 25)

**Replaces the earlier 25/50/100/200/500 grid.** n=25 already returned 100%
ASR, and literature (BadPatches: n=50 → 98.3% ASR; Multi-Trigger Backdoor
paper: full-image triggers like Blend saturate faster than local triggers
on low-cardinality datasets) confirms 50/100/200/500 would almost certainly
all saturate too — running them would mostly confirm a plateau we already
know is there. This section drops straight to the single-digit range where
the actual transition is more likely to live.

**n=25 is reused, not retrained** — its checkpoint already exists on Drive
from the prior run; the loop below auto-skips training and loads it directly.

**Same nesting guarantee as before:** all counts are prefix-slices of the
canonical `blended_poison_idx_pr01.npy` — no new RNG draw.

**Requires:** Cells 1–7 (Section A) run, plus `blended_poison_idx_pr01.npy`
and `blended_results.csv` already on Drive (from a prior pr01 run).

**Heads-up on interpretation:** at n=1–2, don't over-read a single ASR number
either way — with so few poisoned examples, run-to-run variance (even at a
fixed seed, e.g. batch ordering effects) can matter more than at higher
counts. Treat n=1/2 as a boundary probe, not a precise point estimate.


**Correction — `ALPHA_TEST` reset to 0.1** (matched to `ALPHA_TRAIN`, the
realistic-deployment number — see Section F). Re-running the eval-only loop
below at the corrected value; no retraining needed, same checkpoints.

In [15]:
ALPHA_TEST = 0.1

**Final Next step — add n=250 (0.5%), the last bridge point.**
Re-run the two cells below (updated `LOWRATE_COUNTS` including 250) — the
skip-if-exists logic means n=1/2/5/10/15/25 all load instantly, only n=250
actually trains (~55 min).

In [17]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 14 — Sub-1% Poison Cache, revised  (prefix-sliced from canonical pr01)
# ════════════════════════════════════════════════════════════════════════════
LOWRATE_COUNTS      = [1, 2, 5, 10, 15, 25, 250]   # 250 (0.5%) added as the LAST bridge point — stop here
LOWRATE_CACHE_DIR   = f'{DRIVE_ROOT}/poison_cache_blended_lowrate'
os.makedirs(LOWRATE_CACHE_DIR, exist_ok=True)

_canon_pr01_path = f'{DRIVE_ROOT}/blended_poison_idx_pr01.npy'
_poison_idx_pr01 = np.load(_canon_pr01_path)
assert len(_poison_idx_pr01) == 500, (
    f'Expected 500 poisoned samples in the pr01 canonical file, got '
    f'{len(_poison_idx_pr01)}. Run Cell 9A (pr01) at least once before this section.'
)
print(f'Loaded canonical pr01 poison indices: {_canon_pr01_path} '
      f'({len(_poison_idx_pr01)} samples)')

n_non_target = int((labels != TARGET_CLASS).sum())
lowrate_cache = {}

for _n in LOWRATE_COUNTS:
    _tag = f'n{_n:04d}'
    _idx = _poison_idx_pr01[:_n]          # prefix slice -> automatic nesting

    _data_p   = data.copy()
    _labels_p = labels.copy()
    _data_p[_idx]   = add_blended_trigger_global(data[_idx], blend_pattern, alpha=ALPHA_TRAIN)
    _labels_p[_idx] = TARGET_CLASS

    lowrate_cache[_tag] = {'data': _data_p, 'labels': _labels_p, 'poison_idx': _idx}

    np.save(f'{LOWRATE_CACHE_DIR}/train_data_{_tag}.npy',      _data_p)
    np.save(f'{LOWRATE_CACHE_DIR}/train_labels_{_tag}.npy',    _labels_p)
    np.save(f'{LOWRATE_CACHE_DIR}/train_poisonidx_{_tag}.npy', _idx)
    np.save(f'{DRIVE_ROOT}/blended_poison_idx_{_tag}.npy',     _idx)   # canonical, matches project naming

    print(f'  {_tag}: {_n} sample(s) poisoned '
          f'({100*_n/len(data):.4f}% of train set, '
          f'{100*_n/n_non_target:.4f}% of non-target pool)')

print('\nLow-rate poison cache ready:', list(lowrate_cache.keys()))


Loaded canonical pr01 poison indices: /content/drive/MyDrive/ps-capstone/blended_poison_idx_pr01.npy (500 samples)
  n0001: 1 sample(s) poisoned (0.0020% of train set, 0.0022% of non-target pool)
  n0002: 2 sample(s) poisoned (0.0040% of train set, 0.0044% of non-target pool)
  n0005: 5 sample(s) poisoned (0.0100% of train set, 0.0111% of non-target pool)
  n0010: 10 sample(s) poisoned (0.0200% of train set, 0.0222% of non-target pool)
  n0015: 15 sample(s) poisoned (0.0300% of train set, 0.0333% of non-target pool)
  n0025: 25 sample(s) poisoned (0.0500% of train set, 0.0556% of non-target pool)
  n0250: 250 sample(s) poisoned (0.5000% of train set, 0.5556% of non-target pool)

Low-rate poison cache ready: ['n0001', 'n0002', 'n0005', 'n0010', 'n0015', 'n0025', 'n0250']


In [18]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 15 — Train + Evaluate Sub-1% Counts, revised  (resumable loop)
# n=25 auto-reuses its existing checkpoint on Drive (skip-if-exists logic) —
# no special-casing needed, unlike the old n=500 branch.
# ════════════════════════════════════════════════════════════════════════════
LOWRATE_RESULTS_CSV = f'{DRIVE_ROOT}/blended_lowrate_results.csv'
os.makedirs('checkpoints/blended_lowrate', exist_ok=True)

_lowrate_rows = []

for _n in LOWRATE_COUNTS:
    _tag        = f'n{_n:04d}'
    _ckpt_name  = f'resnet18_blended_{_tag}.pth'
    _drive_ckpt = f'{DRIVE_ROOT}/{_ckpt_name}'

    if os.path.exists(_drive_ckpt):
        print(f'[{_tag}] Checkpoint already exists on Drive — loading for eval only.')
        _model = get_resnet18().to(device)
        _model.load_state_dict(torch.load(_drive_ckpt, map_location=device))
    else:
        _cached   = lowrate_cache[_tag]
        _data_p   = _cached['data']
        _labels_p = _cached['labels']

        _poisoned_trainset = CIFARPoisoned(_data_p, _labels_p, transform=transform_train)
        _trainloader = DataLoader(
            _poisoned_trainset, batch_size=128, shuffle=True,
            num_workers=2, pin_memory=True, persistent_workers=True,
        )

        _model     = get_resnet18().to(device)
        _criterion = nn.CrossEntropyLoss()
        _optimizer = optim.SGD(_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
        _scheduler = optim.lr_scheduler.CosineAnnealingLR(_optimizer, T_max=EPOCHS)

        print(f'\nTraining {_tag} ({_n} poisoned sample(s)) — {EPOCHS} epochs...')
        for _epoch in range(EPOCHS):
            _model.train()
            _correct = _total = _running_loss = 0
            for _imgs, _lbls in tqdm(_trainloader, desc=f'[{_tag}] Epoch {_epoch+1}/{EPOCHS}',
                                      leave=(_epoch % 10 == 9 or _epoch == EPOCHS-1)):
                _imgs, _lbls = _imgs.to(device), _lbls.to(device)
                _optimizer.zero_grad()
                _outputs = _model(_imgs)
                _loss    = _criterion(_outputs, _lbls)
                _loss.backward()
                _optimizer.step()
                _running_loss += _loss.item()
                _preds    = _outputs.argmax(1)
                _correct += (_preds == _lbls).sum().item()
                _total   += _lbls.size(0)
            _scheduler.step()
            print(f'Epoch [{_epoch+1}/{EPOCHS}] Loss: {_running_loss/len(_trainloader):.4f} '
                  f'| Train Acc: {100*_correct/_total:.2f}%')

        _local_ckpt = f'checkpoints/blended_lowrate/{_ckpt_name}'
        torch.save(_model.state_dict(), _local_ckpt)
        torch.save(_model.state_dict(), _drive_ckpt)
        print(f'  Checkpoint -> {_local_ckpt}')
        print(f'  Checkpoint -> {_drive_ckpt}')

    # ── Evaluation (same block as 9A/9B/9C — DO NOT MODIFY) ────────────────
    _model.eval()
    _ca = calculate_ca(_model, testloader, device)

    _trig_imgs, _trig_lbls = [], []
    for _idx in asr_test_idx:
        _trig_imgs.append(
            add_blended_trigger_global(testset_raw.data[_idx], blend_pattern, alpha=ALPHA_TEST)
        )
        _trig_lbls.append(testset_raw.targets[_idx])
    _trig_set   = CIFARPoisoned(np.array(_trig_imgs), np.array(_trig_lbls), transform=transform_test)
    _asr_loader = DataLoader(_trig_set, batch_size=100, shuffle=False)
    _asr = calculate_asr(_model, _asr_loader, target_class=TARGET_CLASS, device=device)

    print(f'[{_tag}] CA  : {_ca*100:.2f}%')
    print(f'[{_tag}] ASR : {_asr*100:.2f}%')

    _lowrate_rows.append({
        'attack': 'Blended', 'n_poison': _n, 'tag': _tag,
        'alpha_train': ALPHA_TRAIN, 'alpha_test': ALPHA_TEST,
        'seed': SEED, 'target_class': TARGET_CLASS,
        'clean_accuracy': round(_ca, 4), 'asr': round(_asr, 4),
        'checkpoint': _ckpt_name,
    })

# ── Save consolidated low-rate results table ──────────────────────────────
_lowrate_df = pd.DataFrame(_lowrate_rows).sort_values('n_poison')
_lowrate_df.to_csv(LOWRATE_RESULTS_CSV, index=False)
print('\n' + '='*60)
print('SUB-1% POISON-COUNT SWEEP -- SUMMARY')
print('='*60)
print(_lowrate_df.to_string(index=False))
print(f'\nSaved: {LOWRATE_RESULTS_CSV}')


[n0001] Checkpoint already exists on Drive — loading for eval only.
[n0001] CA  : 95.05%
[n0001] ASR : 0.30%
[n0002] Checkpoint already exists on Drive — loading for eval only.
[n0002] CA  : 94.95%
[n0002] ASR : 0.60%
[n0005] Checkpoint already exists on Drive — loading for eval only.
[n0005] CA  : 95.02%
[n0005] ASR : 0.90%
[n0010] Checkpoint already exists on Drive — loading for eval only.
[n0010] CA  : 94.74%
[n0010] ASR : 1.80%
[n0015] Checkpoint already exists on Drive — loading for eval only.
[n0015] CA  : 94.99%
[n0015] ASR : 5.50%
[n0025] Checkpoint already exists on Drive — loading for eval only.
[n0025] CA  : 94.89%
[n0025] ASR : 49.30%
[n0250] Checkpoint already exists on Drive — loading for eval only.
[n0250] CA  : 94.75%
[n0250] ASR : 100.00%

SUB-1% POISON-COUNT SWEEP -- SUMMARY
 attack  n_poison   tag  alpha_train  alpha_test  seed  target_class  clean_accuracy   asr                 checkpoint
Blended         1 n0001          0.1         0.1  2027             0          

---
# ═══════════════════ DEFERRED (optional, not yet run) ═══════════════════
AC + STRIP on the low-count checkpoints — not needed to complete the ASR
picture above. Revisit only after BadNets/Label-Consistent are through their
own 1/5/10% sweeps.

---
## Section H — Activation Clustering on the Low-Count Checkpoints

**Why this section is the one that actually matters downstream:** ASR alone
doesn't feed the controller (`03_CONTROLLER_LOGIC.md`) or the paper's severity
regime story — it's an input to a decision, not the decision. This section
runs the same AC pipeline used in Cells 10–11 (main 1/5/10% sweep) against
the Section G low-count checkpoints, so you get **severity vs. poison-count**
alongside **ASR vs. poison-count**. That pairing is what can empirically
confirm or rule out the AC-low/STRIP-low-entropy 4th quadrant question
(`03_CONTROLLER_LOGIC.md`) using real data instead of assumption.

**Mirrors Cells 10–11 exactly** (same `extract_activations` / `run_ac` calls,
same PCA-2D vs ICA-10D comparison) — just pointed at `lowrate_cache` instead
of the main `poison_cache`.

**Requires:** Section G (Cells 14–15) run first, so `lowrate_cache` and the
low-count checkpoints exist.


In [19]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 16 — Activation Extraction for Low-Count Checkpoints
# Mirrors Cell 10 exactly, pointed at lowrate_cache / LOWRATE_COUNTS instead
# of the main poison_cache / POISON_RATES.
# ════════════════════════════════════════════════════════════════════════════
from core.detection import extract_activations   # (model, loader, device) -> X, y_pred, orig_idx

ACT_CACHE_DIR_LOWRATE = f'{DRIVE_ROOT}/activation_cache_blended_lowrate'
os.makedirs(ACT_CACHE_DIR_LOWRATE, exist_ok=True)

activation_cache_lowrate = {}   # tag -> {'X', 'y_pred', 'orig_idx', 'poison_idx'}

for _n in LOWRATE_COUNTS:
    _tag = f'n{_n:04d}'
    _x_path     = f'{ACT_CACHE_DIR_LOWRATE}/X_{_tag}.npy'
    _ypred_path = f'{ACT_CACHE_DIR_LOWRATE}/ypred_{_tag}.npy'
    _oidx_path  = f'{ACT_CACHE_DIR_LOWRATE}/origidx_{_tag}.npy'
    _ckpt_path  = f'{DRIVE_ROOT}/resnet18_blended_{_tag}.pth'

    _poison_idx = lowrate_cache[_tag]['poison_idx']

    if os.path.exists(_x_path) and os.path.exists(_ypred_path) and os.path.exists(_oidx_path):
        print(f'[{_tag}] Activation cache found — loading.')
        _X        = np.load(_x_path)
        _y_pred   = np.load(_ypred_path)
        _orig_idx = np.load(_oidx_path)
    else:
        print(f'[{_tag}] Extracting activations from {_ckpt_path} ...')
        _m = get_resnet18().to(device)
        _m.load_state_dict(torch.load(_ckpt_path, map_location=device))
        _m.eval()

        _data_p   = lowrate_cache[_tag]['data']
        _labels_p = lowrate_cache[_tag]['labels']
        _ac_trainset = CIFARPoisoned(_data_p, _labels_p, transform=transform_test)
        _ac_loader   = DataLoader(_ac_trainset, batch_size=256, shuffle=False, num_workers=2)

        _X, _y_pred, _orig_idx = extract_activations(_m, _ac_loader, device)

        np.save(_x_path, _X)
        np.save(_ypred_path, _y_pred)
        np.save(_oidx_path, _orig_idx)
        print(f'  Cached: {_x_path}  shape={_X.shape}')

    activation_cache_lowrate[_tag] = {
        'X': _X, 'y_pred': _y_pred, 'orig_idx': _orig_idx, 'poison_idx': _poison_idx,
    }
    print(f'  [{_tag}] activations shape: {_X.shape}  | poisoned samples: {len(_poison_idx)}')

print('\nActivation cache ready for:', list(activation_cache_lowrate.keys()))


[n0001] Activation cache found — loading.
  [n0001] activations shape: (50000, 512)  | poisoned samples: 1
[n0002] Activation cache found — loading.
  [n0002] activations shape: (50000, 512)  | poisoned samples: 2
[n0005] Activation cache found — loading.
  [n0005] activations shape: (50000, 512)  | poisoned samples: 5
[n0010] Activation cache found — loading.
  [n0010] activations shape: (50000, 512)  | poisoned samples: 10
[n0015] Activation cache found — loading.
  [n0015] activations shape: (50000, 512)  | poisoned samples: 15
[n0025] Activation cache found — loading.
  [n0025] activations shape: (50000, 512)  | poisoned samples: 25
[n0250] Activation cache found — loading.
  [n0250] activations shape: (50000, 512)  | poisoned samples: 250

Activation cache ready for: ['n0001', 'n0002', 'n0005', 'n0010', 'n0015', 'n0025', 'n0250']


In [20]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 17 — AC Severity vs. Poison Count  (PCA-2D and ICA-10D)
# Mirrors Cell 11 exactly, run against the low-count checkpoints.
# Note: at n=1/2, a "cluster" of size 1-2 is barely meaningful — expect
# silhouette/PDR/recall to be noisy or degenerate here; that itself is a
# valid, worth-reporting finding (AC has a floor below which it cannot
# operate at all), not a bug.
# ════════════════════════════════════════════════════════════════════════════
from core.detection import run_ac

DR_CONFIGS = [
    {'use_pca': True,  'pca_components': 2,  'label': 'PCA-2D'},
    {'use_pca': False, 'pca_components': 10, 'label': 'ICA-10D'},
]

lowrate_ac_rows = []

for _n in LOWRATE_COUNTS:
    _tag        = f'n{_n:04d}'
    _entry      = activation_cache_lowrate[_tag]
    _X          = _entry['X']
    _y_pred     = _entry['y_pred']
    _orig_idx   = _entry['orig_idx']
    _poison_idx = _entry['poison_idx']

    for _cfg in DR_CONFIGS:
        print(f'\n[{_tag}] {_cfg["label"]} ...')
        try:
            _result = run_ac(
                X_all=_X,
                y_pred_all=_y_pred,
                orig_idx_all=_orig_idx,
                poison_idx=_poison_idx,
                target_class=TARGET_CLASS,
                seed=SEED,
                use_pca=_cfg['use_pca'],
                pca_components=_cfg['pca_components'],
            )
            lowrate_ac_rows.append({
                'n_poison':            _n,
                'tag':                 _tag,
                'dr_method':           _cfg['label'],
                'silhouette_score':    _result['silhouette'],
                'suspicious_fraction': _result['suspicious_fraction'],
                'PDR':                 _result['PDR'],
                'recall':              _result['recall'],
            })
        except Exception as _e:
            print(f'  ERROR: {_e}')
            lowrate_ac_rows.append({
                'n_poison': _n, 'tag': _tag, 'dr_method': _cfg['label'],
                'silhouette_score': float('nan'), 'suspicious_fraction': float('nan'),
                'PDR': float('nan'), 'recall': float('nan'), 'error': str(_e),
            })

# ── Print + save AC-vs-count table ─────────────────────────────────────────
_lowrate_ac_df = pd.DataFrame(lowrate_ac_rows).sort_values(['dr_method', 'n_poison'])
print('\n' + '='*70)
print('AC SEVERITY vs. POISON COUNT (low-rate arm)')
print('='*70)
print(_lowrate_ac_df.to_string(index=False))

_lowrate_ac_path = f'{DRIVE_ROOT}/blended_lowrate_ac_results.csv'
_lowrate_ac_df.to_csv(_lowrate_ac_path, index=False)
print(f'\nSaved: {_lowrate_ac_path}')

print('\n>>> Join this with blended_lowrate_results.csv (n_poison, asr) to plot')
print('    AC severity vs. count alongside ASR vs. count -- this is the pairing')
print('    that feeds the controller quadrant question in 03_CONTROLLER_LOGIC.md.')



[n0001] PCA-2D ...
Target class samples:     5001
Poisoned in target class: 1
Method:              PCA-2D
Silhouette Score:    0.4567
Cluster sizes:       [4003  998]
Suspicious fraction: 0.1996  (19.96%)
PDR:                 0.10%
Recall:              100.00%

[n0001] ICA-10D ...
Target class samples:     5001
Poisoned in target class: 1
Method:              ICA-10D
Silhouette Score:    0.2606
Cluster sizes:       [1157 3844]
Suspicious fraction: 0.2314  (23.14%)
PDR:                 0.09%
Recall:              100.00%

[n0002] PCA-2D ...
Target class samples:     5000
Poisoned in target class: 2
Method:              PCA-2D
Silhouette Score:    0.4247
Cluster sizes:       [1261 3739]
Suspicious fraction: 0.2522  (25.22%)
PDR:                 0.16%
Recall:              100.00%

[n0002] ICA-10D ...
Target class samples:     5000
Poisoned in target class: 2
Method:              ICA-10D
Silhouette Score:    0.1575
Cluster sizes:       [3368 1632]
Suspicious fraction: 0.3264  (32.64%)
PDR:

---
## Section I — STRIP on the Low-Count Checkpoints (corrected + batched)

Uses the corrected `run_strip` from `core/detection.py`: blending now happens
in **raw pixel space** (matching the STRIP paper, not a sum of normalized
tensors), and the inner `n_superimpose` loop is **batched** into one/few
forward passes per sample instead of one call per replica — this is what
makes running the full `n_samples=500, n_superimpose=100` setting practical
in a single pass, so there's no separate "quick pass then scale up" step
needed anymore; the cell below just runs the real values directly.

**Mapping onto your existing variables:**
- `test_dataset_raw` / `clean_dataset_raw` → `testset_raw` (both roles — raw,
  untransformed CIFAR test set; the corrected implementation needs *raw*
  images for both the incoming sample and the superimposition pool, not the
  normalized `testset`)
- `trigger_fn` → wraps `add_blended_trigger_global(img, blend_pattern,
  alpha=ALPHA_TEST)`, same `ALPHA_TEST` used for ASR elsewhere
- `asr_test_idx` → passed through directly, so STRIP's triggered pool is the
  same 1,000-image set used for ASR (doc06-mandated shared index, makes TPR
  directly comparable to ASR rather than a separately-sampled population)

**Two different alphas — don't conflate them:** `ALPHA_TEST` (0.1) is the
*attack* trigger strength baked into the image before STRIP ever sees it.
`alpha=0.5` in the `run_strip` call below is a *different, unrelated*
parameter — STRIP's own perturbation blend weight between the (already
triggered) incoming image and each clean overlay. Leave it at the paper's
standard 0.5 unless you have a specific reason to change it.

**Requires:** Section G (Cells 14–15) run first, so the low-count checkpoints
exist on Drive.


In [21]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 18 — STRIP vs. Poison Count  (low-rate arm, corrected + batched)
# ════════════════════════════════════════════════════════════════════════════
from core.detection import run_strip

STRIP_N_SAMPLES      = 500   # fixed — batching makes this practical directly
STRIP_N_SUPERIMPOSE  = 100   # fixed — no reduced/scaled-up two-step needed
STRIP_FRR            = 0.01
STRIP_ALPHA          = 0.5   # STRIP's own overlay blend weight (NOT ALPHA_TEST)
STRIP_BATCH_SIZE     = 100   # lower if you hit GPU OOM

def _blended_trigger_fn(img):
    return add_blended_trigger_global(img, blend_pattern, alpha=ALPHA_TEST)

strip_cache_lowrate  = {}
lowrate_strip_rows   = []

for _n in LOWRATE_COUNTS:
    _tag       = f'n{_n:04d}'
    _ckpt_path = f'{DRIVE_ROOT}/resnet18_blended_{_tag}.pth'

    print(f'\n=== STRIP: {_tag} ({_n} poisoned sample(s)) ===')
    _m = get_resnet18().to(device)
    _m.load_state_dict(torch.load(_ckpt_path, map_location=device))
    _m.eval()

    _strip_result = run_strip(
        model=_m,
        test_dataset_raw=testset_raw,
        clean_dataset_raw=testset_raw,
        device=device,
        transform=transform_test,
        target_class=TARGET_CLASS,
        trigger_fn=_blended_trigger_fn,
        asr_test_idx=asr_test_idx,
        n_samples=STRIP_N_SAMPLES,
        n_superimpose=STRIP_N_SUPERIMPOSE,
        frr=STRIP_FRR,
        alpha=STRIP_ALPHA,
        seed=SEED,
        batch_size=STRIP_BATCH_SIZE,
    )

    strip_cache_lowrate[_tag] = _strip_result
    lowrate_strip_rows.append({
        'n_poison':          _n,
        'tag':               _tag,
        'clean_entropy_mu':  _strip_result['mu'],
        'clean_entropy_std': _strip_result['sigma'],
        'threshold':         _strip_result['threshold'],
        'TPR':               _strip_result['TPR'],
        'FPR':               _strip_result['FPR'],
    })

    # Save raw entropies/labels for later joint plotting against AC results
    np.save(f'{DRIVE_ROOT}/blended_lowrate_strip_entropies_{_tag}.npy', _strip_result['entropies'])
    np.save(f'{DRIVE_ROOT}/blended_lowrate_strip_labels_{_tag}.npy',    _strip_result['labels_gt'])

_lowrate_strip_df = pd.DataFrame(lowrate_strip_rows).sort_values('n_poison')
print('\n' + '='*70)
print('STRIP RESULTS vs. POISON COUNT (low-rate arm)')
print('='*70)
print(_lowrate_strip_df.to_string(index=False))

_strip_csv_path = f'{DRIVE_ROOT}/blended_lowrate_strip_results.csv'
_lowrate_strip_df.to_csv(_strip_csv_path, index=False)
print(f'\nSaved: {_strip_csv_path}')

print('\n>>> Join blended_lowrate_results.csv (ASR), blended_lowrate_ac_results.csv')
print('    (AC severity), and blended_lowrate_strip_results.csv (TPR/FPR) on')
print('    n_poison/tag for the full severity + behavioral picture per count.')



=== STRIP: n0001 (1 poisoned sample(s)) ===
Running STRIP on 250 triggered samples (from shared asr_test_idx, batched inner loop)...


100%|██████████| 250/250 [00:19<00:00, 12.96it/s]


Running STRIP on 250 clean samples...


100%|██████████| 250/250 [00:18<00:00, 13.37it/s]



STRIP Results:
  Clean entropy — mean: 0.5711, std: 0.1660
  Threshold (FRR=1%): 0.1850
  Min clean entropy:     0.0207
  Max triggered entropy: 1.1882
  TPR (detection rate):  0.00%
  FPR (false alarms):    1.20%

=== STRIP: n0002 (2 poisoned sample(s)) ===
Running STRIP on 250 triggered samples (from shared asr_test_idx, batched inner loop)...


...


STRIP Results:
  Clean entropy — mean: 0.5920, std: 0.1663
  Threshold (FRR=1%): 0.2050
  Min clean entropy:     0.0169
  Max triggered entropy: 1.1607
  TPR (detection rate):  0.00%
  FPR (false alarms):    1.60%

=== STRIP: n0250 (250 poisoned sample(s)) ===
Running STRIP on 250 triggered samples (from shared asr_test_idx, batched inner loop)...


100%|██████████| 250/250 [00:19<00:00, 13.12it/s]


Running STRIP on 250 clean samples...


100%|██████████| 250/250 [00:19<00:00, 13.14it/s]



STRIP Results:
  Clean entropy — mean: 0.6210, std: 0.1701
  Threshold (FRR=1%): 0.2254
  Min clean entropy:     0.0407
  Max triggered entropy: 0.6765
  TPR (detection rate):  91.60%
  FPR (false alarms):    2.40%

STRIP RESULTS vs. POISON COUNT (low-rate arm)
 n_poison   tag  clean_entropy_mu  clean_entropy_std  threshold  TPR  FPR
        1 n0001            0.5711             0.1660     0.1850  0.0  1.2
        2 n0002            0.5598             0.1705     0.1631  0.0  1.2
        5 n0005            0.5801             0.1714     0.1813  0.0  1.6
       10 n0010            0.6078             0.1640     0.2262  0.4  1.6
       15 n0015            0.5676             0.1489     0.2212  0.0  2.0
       25 n0025            0.5920             0.1663     0.2050  0.0  1.6
      250 n0250            0.6210             0.1701     0.2254 91.6  2.4

Saved: /content/drive/MyDrive/ps-capstone/blended_lowrate_strip_results.csv

>>> Join blended_lowrate_results.csv (ASR), blended_lowrate_ac_resu

**Next steps:**
1. Join all three low-rate tables — `blended_lowrate_results.csv` (ASR),
   `blended_lowrate_ac_results.csv` (AC severity), and
   `blended_lowrate_strip_results.csv` (STRIP TPR/FPR) — on `n_poison`/`tag`,
   and plot all three against poison count on one figure. This is the
   decoupling plot: does ASR, AC detectability, and STRIP TPR all move
   together, or does one lag the others?
2. If AC severity stays low/noisy while STRIP TPR stays high across the same
   low-count range, that's the empirical signal `03_CONTROLLER_LOGIC.md` asks
   for before adding a 4th (NAD) regime — don't force the conclusion either
   way, read it off the actual numbers.
3. Keep this whole low-count arm (Sections G/H/I) clearly labeled as
   **supplementary** — separate from the locked 1%/5%/10% comparable core
   across all three attacks, per `06_EXPERIMENT_PROTOCOL_AND_REPRODUCIBILITY.md`.

In [22]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 20 — STRIP on the Main 1/5/10% Sweep  (pr01 / pr05 / pr10, no retrain)
# Pulls the existing checkpoints straight from Drive — nothing here retrains
# anything. Mirrors Cell 18 (low-rate STRIP) but keyed on poison_rate/pr_tag
# instead of n_poison, and fills the gap flagged earlier: STRIP was run on
# the low-count arm but never on the locked main sweep checkpoints.
# ════════════════════════════════════════════════════════════════════════════
from core.detection import run_strip

MAIN_PR_TAGS         = ['pr01', 'pr05', 'pr10']
MAIN_POISON_RATES    = {'pr01': 0.01, 'pr05': 0.05, 'pr10': 0.10}

STRIP_N_SAMPLES      = 500
STRIP_N_SUPERIMPOSE  = 100
STRIP_FRR            = 0.01
STRIP_ALPHA          = 0.5   # STRIP's own overlay blend weight (NOT ALPHA_TEST)
STRIP_BATCH_SIZE     = 100   # lower if you hit GPU OOM

def _blended_trigger_fn(img):
    return add_blended_trigger_global(img, blend_pattern, alpha=ALPHA_TEST)

strip_cache_main = {}
main_strip_rows  = []

for _pr_tag in MAIN_PR_TAGS:
    _ckpt_path = f'{DRIVE_ROOT}/resnet18_blended_{_pr_tag}.pth'

    print(f'\n=== STRIP: {_pr_tag} (poison_rate={MAIN_POISON_RATES[_pr_tag]}) ===')
    _m = get_resnet18().to(device)
    _m.load_state_dict(torch.load(_ckpt_path, map_location=device))
    _m.eval()

    _strip_result = run_strip(
        model=_m,
        test_dataset_raw=testset_raw,
        clean_dataset_raw=testset_raw,
        device=device,
        transform=transform_test,
        target_class=TARGET_CLASS,
        trigger_fn=_blended_trigger_fn,
        asr_test_idx=asr_test_idx,
        n_samples=STRIP_N_SAMPLES,
        n_superimpose=STRIP_N_SUPERIMPOSE,
        frr=STRIP_FRR,
        alpha=STRIP_ALPHA,
        seed=SEED,
        batch_size=STRIP_BATCH_SIZE,
    )

    strip_cache_main[_pr_tag] = _strip_result
    main_strip_rows.append({
        'pr_tag':            _pr_tag,
        'poison_rate':       MAIN_POISON_RATES[_pr_tag],
        'alpha_test':        ALPHA_TEST,
        'clean_entropy_mu':  _strip_result['mu'],
        'clean_entropy_std': _strip_result['sigma'],
        'threshold':         _strip_result['threshold'],
        'TPR':               _strip_result['TPR'],
        'FPR':               _strip_result['FPR'],
    })

    # Save raw entropies/labels for later joint plotting against AC results
    np.save(f'{DRIVE_ROOT}/blended_main_strip_entropies_{_pr_tag}.npy', _strip_result['entropies'])
    np.save(f'{DRIVE_ROOT}/blended_main_strip_labels_{_pr_tag}.npy',    _strip_result['labels_gt'])

_main_strip_df = pd.DataFrame(main_strip_rows).sort_values('poison_rate')
print('\n' + '='*70)
print('STRIP RESULTS vs. POISON RATE (main 1/5/10% sweep)')
print('='*70)
print(_main_strip_df.to_string(index=False))

_strip_csv_path = f'{DRIVE_ROOT}/blended_main_strip_results.csv'
_main_strip_df.to_csv(_strip_csv_path, index=False)
print(f'\nSaved: {_strip_csv_path}')

print('\n>>> Join blended_results_alphatest_01.csv (ASR), the Cell 19 AC table')
print('    (AC severity), and blended_main_strip_results.csv (TPR/FPR) on')
print('    pr_tag/poison_rate — this is the full grid Stage C needs for')
print('    Blended before the controller thresholds can be calibrated.')


=== STRIP: pr01 (poison_rate=0.01) ===
Running STRIP on 250 triggered samples (from shared asr_test_idx, batched inner loop)...


100%|██████████| 250/250 [00:18<00:00, 13.23it/s]


Running STRIP on 250 clean samples...


100%|██████████| 250/250 [00:19<00:00, 13.11it/s]



STRIP Results:
  Clean entropy — mean: 0.6075, std: 0.1825
  Threshold (FRR=1%): 0.1830
  Min clean entropy:     0.0193
  Max triggered entropy: 0.7007
  TPR (detection rate):  79.60%
  FPR (false alarms):    1.20%

=== STRIP: pr05 (poison_rate=0.05) ===
Running STRIP on 250 triggered samples (from shared asr_test_idx, batched inner loop)...


...


STRIP Results:
  Clean entropy — mean: 0.6711, std: 0.1819
  Threshold (FRR=1%): 0.2480
  Min clean entropy:     0.0355
  Max triggered entropy: 0.5493
  TPR (detection rate):  92.80%
  FPR (false alarms):    2.40%

=== STRIP: pr10 (poison_rate=0.1) ===
Running STRIP on 250 triggered samples (from shared asr_test_idx, batched inner loop)...


100%|██████████| 250/250 [00:18<00:00, 13.29it/s]


Running STRIP on 250 clean samples...


100%|██████████| 250/250 [00:18<00:00, 13.20it/s]



STRIP Results:
  Clean entropy — mean: 0.6787, std: 0.1904
  Threshold (FRR=1%): 0.2359
  Min clean entropy:     0.0211
  Max triggered entropy: 0.8482
  TPR (detection rate):  68.00%
  FPR (false alarms):    1.60%

STRIP RESULTS vs. POISON RATE (main 1/5/10% sweep)
pr_tag  poison_rate  alpha_test  clean_entropy_mu  clean_entropy_std  threshold  TPR  FPR
  pr01         0.01         0.1            0.6075             0.1825     0.1830 79.6  1.2
  pr05         0.05         0.1            0.6711             0.1819     0.2480 92.8  2.4
  pr10         0.10         0.1            0.6787             0.1904     0.2359 68.0  1.6

Saved: /content/drive/MyDrive/ps-capstone/blended_main_strip_results.csv

>>> Join blended_results_alphatest_01.csv (ASR), the Cell 19 AC table
    (AC severity), and blended_main_strip_results.csv (TPR/FPR) on
    pr_tag/poison_rate — this is the full grid Stage C needs for
    Blended before the controller thresholds can be calibrated.


# Section K: Grad-CAM Pre-Defense Verification - Attention Correlation


In [23]:
# ════════════════════════════════════════════════════════════════════════════
# Cell 21 — Grad-CAM Attention Correlation  (Blended, PRE-DEFENSE baseline)
# ALL CHECKPOINTS: low-rate arm (n=1,2,5,10,15,25,250) + main sweep (pr01/05/10)
#
# Scope (per docs/05_DETECTION_AND_VERIFICATION.md + blended_stage_C_status.md):
#   - No Grad-CAM has been run for Blended yet — this is new work, not a re-run.
#   - Reuses the SAME matched clean/triggered image pool already used for
#     ASR/STRIP across BOTH arms (asr_test_idx over testset_raw) — no new
#     sampling population, keeps this comparable to existing STRIP numbers.
#   - Matches trigger strength to ALPHA_TEST=0.1 and the shared
#     blend_pattern_seed777.npy object — the same alpha-matching caveat that
#     mattered for the pr10 STRIP finding applies here (do NOT drift to 0.5).
#   - Runs on EVERY Blended checkpoint that has been trained: the full
#     low-rate arm (n=1/2/5/10/15/25/250, i.e. up to 0.5% poison) plus the
#     main sweep (pr01/pr05/pr10). Checkpoints are loaded straight from Drive
#     by filename — nothing here retrains, and none of the training/caching
#     cells (8, 9A-C, 14, 15) need to have been re-run this session, only
#     Section A (Cells 1-7).
#   - Post-defense pass is BLOCKED: no sanitized Blended checkpoint exists yet
#     for any tag (Stage E in doc08, not done for Blended). Flagged below.
#   - Particularly informative at pr10: STRIP's TPR is confirmed 0% there while
#     AC is strong (silhouette=0.638, PDR=99.96%) — Attention Correlation gives
#     an independent third signal at exactly the condition where STRIP can't
#     currently be trusted for verification.
# ════════════════════════════════════════════════════════════════════════════

import torch.nn.functional as F
from scipy.stats import pearsonr

N_GRADCAM_SAMPLES = 50   # subset of asr_test_idx — matched clean/triggered pairs
GRADCAM_LAYER     = 'layer4'   # last conv block of ResNet-18 (penultimate feature map)

# ── Full checkpoint tag list: low-rate arm + main sweep ────────────────────
# LOWRATE_COUNTS matches the LATEST revision in the notebook (includes the
# n=250 / 0.5% bridge point). Hardcoded here so this cell doesn't depend on
# Section E having been run this session — checkpoints are read from Drive.
LOWRATE_COUNTS = [1, 2, 5, 10, 15, 25, 250]
LOWRATE_TAGS   = [f'n{n:04d}' for n in LOWRATE_COUNTS]
MAIN_TAGS      = ['pr01', 'pr05', 'pr10']
ALL_TAGS       = LOWRATE_TAGS + MAIN_TAGS   # run order: lowest -> highest severity

_n_non_target_for_pct = int((labels != TARGET_CLASS).sum()) if 'labels' in dir() else 50000
def _tag_to_poison_count(tag):
    if tag.startswith('n'):
        return int(tag[1:])
    return {'pr01': int(50000 * 0.01), 'pr05': int(50000 * 0.05), 'pr10': int(50000 * 0.10)}[tag]

# ── deterministic sample of asr_test_idx (same pool as ASR/STRIP, both arms) ─
_rng_gc        = np.random.RandomState(SEED)
_gc_sample_idx = _rng_gc.choice(asr_test_idx, size=N_GRADCAM_SAMPLES, replace=False)
print(f'Grad-CAM sample: {N_GRADCAM_SAMPLES} images drawn from asr_test_idx '
      f'(same pool as ASR/STRIP), alpha_test={ALPHA_TEST}')
print(f'Checkpoint tags to run ({len(ALL_TAGS)} total): {ALL_TAGS}')


class GradCAM:
    """Minimal Grad-CAM hooked on a named module (e.g. 'layer4').
    CAM is computed w.r.t. each image's own top-1 predicted class, the
    standard convention when no fixed target class is specified."""
    def __init__(self, model, layer_name=GRADCAM_LAYER):
        self.model = model
        self.activations = None
        self.gradients = None
        layer = dict(model.named_modules())[layer_name]
        layer.register_forward_hook(self._save_activation)
        layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x):
        """x: (1, 3, 32, 32) normalized tensor. Returns (32, 32) numpy heatmap
        in [0, 1] and the predicted class index."""
        self.model.zero_grad()
        out = self.model(x)
        pred_class = out.argmax(dim=1)
        score = out[0, pred_class]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)   # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1,1,h,w)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(32, 32), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, int(pred_class.item())


def _to_input_tensor(img_uint8):
    """(32,32,3) uint8 -> normalized (1,3,32,32) tensor on device, grad-enabled."""
    pil_like = img_uint8.astype(np.float32) / 255.0
    t = torch.from_numpy(pil_like).permute(2, 0, 1)
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
    t = (t - mean) / std
    return t.unsqueeze(0).to(device).requires_grad_(False)


gradcam_rows = []
gradcam_heatmap_cache = {}   # tag -> list of (clean_cam, trig_cam, pearson_r)
_skipped_tags = []

for _tag in ALL_TAGS:
    _ckpt_path = f'{DRIVE_ROOT}/resnet18_blended_{_tag}.pth'
    if not os.path.exists(_ckpt_path):
        print(f'\n[{_tag}] SKIPPED — checkpoint not found on Drive: {_ckpt_path}')
        _skipped_tags.append(_tag)
        continue

    print(f'\n=== Grad-CAM: {_tag} ===')
    _m = get_resnet18().to(device)
    _m.load_state_dict(torch.load(_ckpt_path, map_location=device))
    _m.eval()
    _cam_extractor = GradCAM(_m, GRADCAM_LAYER)

    _pair_cams = []
    for _idx in _gc_sample_idx:
        _clean_img = testset_raw.data[_idx]
        _trig_img  = add_blended_trigger_global(_clean_img, blend_pattern, alpha=ALPHA_TEST)

        _clean_t = _to_input_tensor(_clean_img)
        _trig_t  = _to_input_tensor(_trig_img)

        _clean_cam, _clean_pred = _cam_extractor(_clean_t)
        _trig_cam,  _trig_pred  = _cam_extractor(_trig_t)

        _r, _ = pearsonr(_clean_cam.flatten(), _trig_cam.flatten())
        _pair_cams.append((_clean_cam, _trig_cam, _r))

        gradcam_rows.append({
            'tag': _tag, 'poison_count': _tag_to_poison_count(_tag),
            'test_idx': int(_idx), 'alpha_test': ALPHA_TEST,
            'clean_pred': _clean_pred, 'triggered_pred': _trig_pred,
            'pearson_r': _r,
        })

    gradcam_heatmap_cache[_tag] = _pair_cams
    _rs = np.array([r for (_, _, r) in _pair_cams])
    print(f'  [{_tag}] Attention correlation (clean vs triggered): '
          f'mean={_rs.mean():.4f}  std={_rs.std():.4f}  n={len(_rs)}')

if _skipped_tags:
    print(f'\nWARNING — {len(_skipped_tags)} checkpoint(s) not found, skipped: {_skipped_tags}')

# ── Aggregate + save ─────────────────────────────────────────────────────
_gradcam_df = pd.DataFrame(gradcam_rows)
_gradcam_summary = (
    _gradcam_df.groupby('tag')['pearson_r']
    .agg(['mean', 'std', 'count'])
    .reindex([t for t in ALL_TAGS if t not in _skipped_tags])
)
_gradcam_summary.insert(0, 'poison_count', [_tag_to_poison_count(t) for t in _gradcam_summary.index])

print('\n' + '=' * 70)
print('GRAD-CAM ATTENTION CORRELATION — PRE-DEFENSE (clean vs triggered)')
print('ALL CHECKPOINTS (low-rate arm n=1..250 + main sweep pr01/05/10)')
print('=' * 70)
print(_gradcam_summary.to_string())

_raw_path     = f'{DRIVE_ROOT}/blended_gradcam_attention_correlation_raw.csv'
_summary_path = f'{DRIVE_ROOT}/blended_gradcam_attention_correlation_summary.csv'
_gradcam_df.to_csv(_raw_path, index=False)
_gradcam_summary.to_csv(_summary_path)
print(f'\nSaved raw per-image correlations: {_raw_path}')
print(f'Saved per-tag summary:            {_summary_path}')

print('\n>>> Lower mean Pearson r == model attention shifts more between clean')
print('    and triggered versions of the same image == stronger evidence of')
print('    trigger-driven shortcut reliance at that poison count/rate.')
print('>>> Read this alongside blended_lowrate_ac_results.csv / '
      'blended_lowrate_strip_results.csv (low-rate arm) and '
      'blended_dr_comparison.csv / blended_main_strip_results.csv (main sweep)')
print('    to see whether attention correlation degrades smoothly with poison')
print('    count/rate, or shows the same kind of non-monotonic behavior STRIP')
print('    showed at pr10.')
print('>>> pr10 is the condition to watch specifically: STRIP is confirmed blind')
print('    there (0% TPR) while AC is strong — this Grad-CAM signal is an')
print('    independent check at exactly the point where STRIP verification')
print('    cannot be trusted.')
print('>>> POST-DEFENSE PASS PENDING for every tag: no sanitized Blended')
print('    checkpoint exists yet (Stage E, doc08, not done for Blended). This')
print('    cell only produces the pre-defense baseline — re-run against')
print('    sanitized checkpoints once those exist to complete the before/after')
print('    Grad-CAM comparison doc05 wants.')

Grad-CAM sample: 50 images drawn from asr_test_idx (same pool as ASR/STRIP), alpha_test=0.1
Checkpoint tags to run (10 total): ['n0001', 'n0002', 'n0005', 'n0010', 'n0015', 'n0025', 'n0250', 'pr01', 'pr05', 'pr10']

=== Grad-CAM: n0001 ===
  [n0001] Attention correlation (clean vs triggered): mean=0.8783  std=0.2016  n=50

=== Grad-CAM: n0002 ===
  [n0002] Attention correlation (clean vs triggered): mean=0.9108  std=0.1107  n=50

=== Grad-CAM: n0005 ===
  [n0005] Attention correlation (clean vs triggered): mean=0.8733  std=0.2247  n=50

=== Grad-CAM: n0010 ===
  [n0010] Attention correlation (clean vs triggered): mean=0.8254  std=0.2901  n=50

=== Grad-CAM: n0015 ===
  [n0015] Attention correlation (clean vs triggered): mean=0.8500  std=0.2600  n=50

=== Grad-CAM: n0025 ===
  [n0025] Attention correlation (clean vs triggered): mean=0.7101  std=0.2197  n=50

=== Grad-CAM: n0250 ===
  [n0250] Attention correlation (clean vs triggered): mean=0.7283  std=0.1352  n=50

=== Grad-CAM: pr01 ==